## Install and Import Libraries

In [ ]:
# Install required packages (silent)
!pip install -q datasets evaluate scikit-learn > /dev/null 2>&1
!pip install -q -U bitsandbytes transformers > /dev/null 2>&1
!pip install -q -U peft > /dev/null 2>&1

## Few-Shot Learning

### Silma

#### 1. Task 1

In [ ]:
rm -rf /content/huggingface_tokenizers_cache /content/outputs /content/unsloth_compiled_cache

In [ ]:
task = 2
label_col = "final_QT"
x_col = "question"
if task == 2:
    label_col = "final_AS"
    x_col = "answer"
elif task == 3:
    label_col = "final_AS"
    x_col = "question"

In [ ]:
import os
import pandas as pd
import torch
import random
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
)
from transformers.optimization import get_linear_schedule_with_warmup
MODEL_ID = "Qwen/Qwen3-8B"   # use "UBC-NLP/MARBERTv2" if you prefer v2
MAX_LEN = 256
VAL_SIZE = 50                  # exactly 50 samples in validation
NUM_LABELS = 3                 # ['1','2','3'] -> 3-way multi-label
# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "/content"
TASK1_FILE = "/content/Train_Dev.tsv"
FEWSHOT_FILE = "/content/dev_data.tsv"
FEWSHOT_LABELS_FILE = "/content/dev_label.tsv"

In [ ]:
# Load Train_Dev.tsv
df_full = pd.read_csv(TASK1_FILE, sep='\t')
df_full.columns = df_full.columns.str.strip()
# df_full['normalized_q'] = df_full['question'].apply(normalize)

print(f"✅ Loaded Train_Dev.tsv with {len(df_full)} rows.")


✅ Loaded Train_Dev.tsv with 350 rows.


In [ ]:
# mkdir with os
os.makedirs("working", exist_ok=True)

In [ ]:
# FEWSHOT_FILE = '/content/working/dev_data.tsv'        # writable location
# FEWSHOT_LABELS_FILE = '/content/working/dev_label.tsv'  # writable location

# # Sample 10 few-shot examples from the full dataset
# fewshot_sample = df_full.sample(n=10, random_state=SEED)

# # Save just the questions (without header) to dev_data.tsv
# fewshot_sample[['answer']].to_csv(FEWSHOT_FILE, sep='\t', index=False, header=False)

# # Save the corresponding labels (without header) to dev_label.tsv
# fewshot_sample[['final_AS']].to_csv(FEWSHOT_LABELS_FILE, sep='\t', index=False, header=False)

# print("✅ Saved 10 few-shot answers + labels to /content/working/")


In [ ]:
# fewshot_sample

In [ ]:
import ast, random, pandas as pd

SEED = 42
random.seed(SEED)

FEWSHOT_FILE        = "/content/working/dev_data.tsv"
FEWSHOT_LABELS_FILE = "/content/working/dev_label.tsv"

# -----------------------------
# helper to normalise a label list
# -----------------------------
def norm_labels(label_cell):
    """Return tuple of sorted label‑ids, e.g. ('1','3')."""
    if isinstance(label_cell, str):
        label_cell = ast.literal_eval(label_cell)
    return tuple(sorted(label_cell))

# -----------------------------
# 1.  build a map  combo → row indices
# -----------------------------
combo2idx = {}
for idx, row in df_full.iterrows():
    combo = norm_labels(row[label_col])
    combo2idx.setdefault(combo, []).append(idx)

# -----------------------------
# 2.  pick ONE example per combo
# -----------------------------
chosen_idx = [
    random.choice(idxs)        # one random row for this combo
    for combo, idxs in combo2idx.items()
]

fewshot_sample = df_full.loc[chosen_idx].copy()

# -----------------------------
# 3.  (optional) pad to 10 rows
# -----------------------------
# TARGET_N = 10
# if len(fewshot_sample) < TARGET_N:
#     remaining = df_full.drop(fewshot_sample.index)
#     extra = remaining.sample(
#         n=TARGET_N - len(fewshot_sample),
#         random_state=SEED
#     )
#     fewshot_sample = pd.concat([fewshot_sample, extra])

# -----------------------------
# 4.  save TSVs
# -----------------------------
fewshot_sample[[x_col]].to_csv(
    FEWSHOT_FILE,
    sep="\t", index=False, header=False
)
fewshot_sample[[label_col]].to_csv(
    FEWSHOT_LABELS_FILE,
    sep="\t", index=False, header=False
)

print(f"✅ Saved {len(fewshot_sample)} few‑shot question + labels covering "
      f"{len(combo2idx)} distinct label‑combinations ➜ /content/working/")


✅ Saved 7 few‑shot question + labels covering 7 distinct label‑combinations ➜ /content/working/


In [ ]:
fewshot_sample

,question,answer,final_QT,final_AS
212,ودي استفسر بخصوص بروزاك 20 كتبه لي اخصائي نفسي...,يجب أن تتابع مع الطبيب كما يجب أن تتابع مع معا...,"['B', 'D']","['1', '2']"
120,ماهو افضل علاج دوائي للاكتئاب وماهي اعراضه الج...,عليك الذهاب للطبيب لكي يصف العلاج.تناولي اقراص...,['B'],['2']
23,اعاني من تخيلات غريبه اثناء النوم حيث استيقظ ت...,هي حالة من اضطرابات النوم على شكل احلام مزعجة ...,"['A', 'D']",['1']
228,تزداد ضربات دقات القلب فوق ال 100 عند الوقوف ل...,سلامتك يجب مراجعة طبيب قلب واذا تاكد انه الوضع...,"['A', 'C']","['1', '3']"
38,صداع توتر قلق الم بل المعده تفكير 24ساعه صعوبه...,اهلا عزيزي اقترح عليك ان تذهب الى طبيب نفسي ل...,"['A', 'B', 'D']","['2', '3']"
101,أحاول منذ 4 سنوات أو أكثر العمل على ذاتي ولكني...,سلامتك، توصفين بكلماتك شكل من اشكال الاكتئاب و...,['A'],"['1', '2', '3']"
57,انا غير راضية عن نفسي من نايحة المظهر اصبحت لد...,"لاتكترثي لذلك لست أنت من صنعت شكلك, لكن هناك أ...",['E'],['3']


In [ ]:
fewshot_sample[[x_col]].to_csv(
    FEWSHOT_FILE,
    sep="\t", index=False, header=False
)
fewshot_sample[[label_col]].to_csv(
    FEWSHOT_LABELS_FILE,
    sep="\t", index=False, header=False
)

print(f"✅ Saved {len(fewshot_sample)} few‑shot question + labels covering "
      f"{len(combo2idx)} distinct label‑combinations ➜ /content/working/")

In [ ]:
# Load the just-written dev_data.tsv (no header)
fewshot_qs = pd.read_csv(FEWSHOT_FILE, sep='\t', header=None, names=[x_col])
# fewshot_qs['normalized_q'] = fewshot_qs['question'].apply(normalize)

print(f"✅ Loaded {len(fewshot_qs)} few-shot examples.")

# Exclude few-shot examples from the full dataset using normalized text
df_filtered = df_full[~df_full[x_col].isin(fewshot_qs[x_col])].reset_index(drop=True)

print(f"📊 Remaining examples after exclusion: {len(df_filtered)} (original: {len(df_full)})")



✅ Loaded 7 few-shot examples.
📊 Remaining examples after exclusion: 343 (original: 350)


In [ ]:
fewshot_sample

,question,answer,final_QT,final_AS
212,ودي استفسر بخصوص بروزاك 20 كتبه لي اخصائي نفسي...,يجب أن تتابع مع الطبيب كما يجب أن تتابع مع معا...,"['B', 'D']","['1', '2']"
120,ماهو افضل علاج دوائي للاكتئاب وماهي اعراضه الج...,عليك الذهاب للطبيب لكي يصف العلاج.تناولي اقراص...,['B'],['2']
23,اعاني من تخيلات غريبه اثناء النوم حيث استيقظ ت...,هي حالة من اضطرابات النوم على شكل احلام مزعجة ...,"['A', 'D']",['1']
228,تزداد ضربات دقات القلب فوق ال 100 عند الوقوف ل...,سلامتك يجب مراجعة طبيب قلب واذا تاكد انه الوضع...,"['A', 'C']","['1', '3']"
38,صداع توتر قلق الم بل المعده تفكير 24ساعه صعوبه...,اهلا عزيزي اقترح عليك ان تذهب الى طبيب نفسي ل...,"['A', 'B', 'D']","['2', '3']"
101,أحاول منذ 4 سنوات أو أكثر العمل على ذاتي ولكني...,سلامتك، توصفين بكلماتك شكل من اشكال الاكتئاب و...,['A'],"['1', '2', '3']"
57,انا غير راضية عن نفسي من نايحة المظهر اصبحت لد...,"لاتكترثي لذلك لست أنت من صنعت شكلك, لكن هناك أ...",['E'],['3']


In [ ]:
df_filtered.head()

,question,answer,final_QT,final_AS
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,سد تى نعم ان هناك العديد من القلق النفسى المرت...,"['A', 'D']","['1', '2']"
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,يجب العرض علي طبيب اختصاصي امراض نفسية سوف يقو...,"['B', 'E']",['2']
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,الوسواس القهري هو من الامراض الشاءعة ويكون في ...,"['A', 'E']","['1', '2']"
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,العلاج النفسي المعرفي السلوكي يعطي نتائج جيدة ...,"['B', 'D']",['1']
4,هل الإحساس بقرب الاجل و الخوف من الموت و الاحل...,نعم بالاضافه للكثير من الاعراض الاخرى الطبيب ا...,"['A', 'B', 'D']","['1', '2']"


In [ ]:
x = df_full[label_col].loc[2]

In [ ]:
label_col

'final_AS'

In [ ]:
print(x)

['1', '2']


In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# os.environ["WANDB_MODE"] = "disabled"; os.environ["WANDB_DISABLED"] = "true"


In [ ]:
df_full = df_filtered

In [ ]:
from sklearn.model_selection import train_test_split

# train_df, val_df = train_test_split(
#     df_full,
#     test_size=0.2,
#     random_state=SEED,
#     stratify=None
# )
val_df = df_full.sample(n=VAL_SIZE, random_state=SEED)
train_df = df_full.drop(val_df.index).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"✅ Training set: {len(train_df)}")
print(f"✅ Validation set: {len(val_df)}")


✅ Training set: 293
✅ Validation set: 50


In [ ]:
import os, ast, random, numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score, jaccard_score

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)


In [ ]:
NUM_LABELS = 3

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
def parse_label_list(cell):
    if isinstance(cell, list):
        return [str(x) for x in cell]
    if isinstance(cell, str):
        try:
            obj = ast.literal_eval(cell)
            if isinstance(obj, (list, tuple)):
                return [str(x) for x in obj]
        except Exception:
            pass
        # fallback: capture tokens made of letters or digits
        return re.findall(r"[A-Za-z0-9]+", cell)
    return []

In [ ]:
import json
# ---- discover the label space from your full dataframe ----
all_labels = set()
for v in df_full[label_col]:
    all_labels.update(parse_label_list(v))

LABEL_SPACE = sorted(all_labels)          # e.g., ['A','B','C','D','E','F','Z']
label2id = {lab: i for i, lab in enumerate(LABEL_SPACE)}
id2label = {i: lab for lab, i in label2id.items()}
NUM_LABELS = len(LABEL_SPACE)

# (optional) persist to reuse at inference time
with open("label_space.json", "w") as f:
    json.dump({"LABEL_SPACE": LABEL_SPACE}, f)

In [ ]:

NUM_LABELS

3

In [ ]:
pip install -q unsloth weave

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.9/307.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.5/612.5 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.7/182.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.1/888.1 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.3 MB/s eta 0:

In [ ]:
import torch
from unsloth import FastLanguageModel
import weave; weave.init('think_test')



SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ====== REST OF SCRIPT ======
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments


max_seq_length = 2048
dtype = None
load_in_4bit = False
MODEL_NAME = "silma-ai/SILMA-9B-Instruct-v1.0"
SAVE_DIR = "lora_model"


# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    force_download=True,
    load_in_4bit=load_in_4bit,
)





/tmp/ipython-input-3080606062.py:2: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


  if event.key is 'enter':

  from pkg_resources import resource_stream, resource_exists

weave: Please login to Weights & Biases (https://wandb.ai/) to continue...


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

RuntimeError: Unsloth: The tokenizer `silma-ai/SILMA-9B-Instruct-v1.0`
does not have a {% if add_generation_prompt %} for generation purposes.
Please file a bug report to the maintainers of `silma-ai/SILMA-9B-Instruct-v1.0` - thanks!

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "silma-ai/SILMA-9B-Instruct-v1.0"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype="bfloat16")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, quantization_config=bnb, device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        if input_text.strip():
            user_message = f"{instruction}\n\n{input_text}"
        else:
            user_message = instruction
        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )
        texts.append(text)
    return {"text": texts}


# dataset = load_dataset("yahma/alpaca-cleaned", split="train")
# half_len = len(dataset) // 10
# dataset = dataset.select(range(half_len))
# dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=2)


In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 5176
})

In [ ]:
print(dataset[300]['text'])

<|im_start|>user
Generate a story with the given keywords.

princess, dragon, knight<|im_end|>
<|im_start|>assistant
<think>

</think>

Once upon a time, in a land far, far away, there was a brave knight named Sir Cedric. Sir Cedric was known throughout the kingdom as a champion of the people; he was kind, just, and always did what was right. One day, Sir Cedric received word that a princess had been kidnapped by a fierce dragon and was being held in a tower on the outskirts of the kingdom. The king of the land called upon Sir Cedric to rescue the princess and bring her back to safety.

Sir Cedric donned his armor, mounted his trusty steed, and set out on his quest to rescue the princess. He rode for many days and many nights until he finally reached the tower where the princess was being held. There, he saw the dragon, perched atop the tower, guarding its prize. Sir Cedric drew his sword and charged forward, ready to do battle with the beast.

The battle was long and fierce, but Sir C

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
# ---- 3) LoRA config (PEFT) ----
lora_cfg = LoraConfig(
    r=16,                       # rank (↑ increases trainable params)
    lora_alpha=16,              # scaling (doesn't change param count)
    lora_dropout=0.0,           # match your Unsloth setting; 0.05 is safer for overfit
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
    # use_gradient_checkpointing="unsloth",
    # random_state=SEED,
    loftq_config=None,
    # use_rslora=True,          # optional (requires recent PEFT); keeps param count same
)

In [ ]:
model = get_peft_model(model, lora_cfg)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)


AttributeError: 'Gemma2ForCausalLM' object has no attribute '_saved_temp_tokenizer'

In [ ]:
import gc
gc.collect()

0

In [ ]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        if input_text.strip():
            user_message = f"{instruction}\n\n{input_text}"
        else:
            user_message = instruction
        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )
        texts.append(text)
    return {"text": texts}




In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
half_len = len(dataset) // 2
dataset = dataset.select(range(half_len))


In [ ]:
print(dataset[5])

{'output': 'The Commodore 64 was a highly successful 8-bit home computer manufactured by Commodore Business Machine (CBM) in 1982, with sales amounting to approximately 17 million units sold between 1983-1986. It dominated the market with between 30% and 40% share and outsold its competitors, including IBM PC clones, Apple Computers, and Atari computers. At its peak, CBM was building 400,000 C64s a month for a couple of years.', 'input': '', 'instruction': 'Write a concise summary of the following:\n"Commodore 64 (commonly known as the C64 or CBM 64) was manufactured by Commodore Business Machine (CBM) in August 1982 with a starting price of $595. It was an 8-bit home computer with remarkable market success. Between 1983-1986, C64 sales amounted to about 17 million units sold, making them the best-selling single personal computer model of all time in 1983-1986. \n\nAdditionally, the Commodore 64 dominated the market with between 30% and 40% share and 2 million units sold per year, outs

In [ ]:
import os, ast, re, random
from typing import List, Dict
import numpy as np, pandas as pd, torch
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments


In [ ]:
def parse_labels(cell) -> List[str]:
    if isinstance(cell, list): return [str(x) for x in cell]
    if isinstance(cell, str):
        try:
            obj = ast.literal_eval(cell)
            if isinstance(obj, (list, tuple)): return [str(x) for x in obj]
        except Exception:
            pass
        return re.findall(r"[123]", cell)
    return []


ID2AR = {"1": "معلومات", "2": "إرشاد مباشر", "3": "دعم عاطفي"}
AR2ID = {v: k for k, v in ID2AR.items()}

def ids_to_ar_string(ids):
    """['1','3'] → 'معلومات، دعم عاطفي' (Arabic comma)."""
    ordered = [ID2AR[i] for i in sorted({str(i) for i in ids}) if i in ID2AR]
    return "، ".join(ordered)



In [ ]:
AR_SYS = (
    "أنت مُصنِّف لأساليب ردود الأطباء. مهمتك تحديد الأسلوب/الأساليب التي يندرج تحتها ردُّ الطبيب،"
    " وقد تنطبق أكثر من فئة في الوقت نفسه.\n"
    "الفئات المتاحة:\n"
    "معلومات (Information): يشمل الإجابات التي تقدّم معلومات أو موارد أو توضيحات، "
    "ويشمل أيضاً طلبات الحصول على معلومات.\n"
    "إرشاد مباشر (Direct Guidance): يشمل الاقتراحات أو التعليمات أو النصائح، "
    "ويشمل الإجابات التي تخبر السائل ماذا ينبغي أن يفعل.\n"
    "دعم عاطفي (Emotional Support): يشمل المواساة أو الطمأنة أو عبارات الدعم والمساندة.\n\n"
    "التعليمات: أعد أسماء الفئات بالعربية تمامًا كما كُتبت أعلاه، مفصولة بـ «،» فقط "
    "دون أي نص إضافي."
)

def build_messages_for_answer(answer_text: str):
    user = (
        f"ردّ الطبيب:\n{answer_text}\n\n"
        "حدد أسماء الفئات فقط كما طُلِب أعلاه:"
    )
    return [
        {"role": "system", "content": AR_SYS},
        {"role": "user",   "content": user},
    ]

In [ ]:
def build_fewshot_messages_all_ar(answer_text: str, shots_df: pd.DataFrame) -> List[Dict[str,str]]:
    """
    system: تعليمات عربية
    ثم كل أمثلة fewshot_sample: (user: ردّ الطبيب ... | assistant: 'معلومات، ...')
    ثم المثال الهدف (user فقط)
    """
    messages = [{"role":"system", "content": AR_SYS}]
    # جميع الأمثلة بدون sampling
    for _, r in shots_df.iterrows():
        ex_ar = ids_to_ar_string(parse_labels(r["final_AS"]))
        usr   = f"ردّ الطبيب:\n{r['answer']}\n\nحدد أسماء الفئات فقط كما طُلِب أعلاه:"
        messages.append({"role":"user", "content": usr})
        messages.append({"role":"assistant", "content": ex_ar})
    # الهدف
    tgt = f"ردّ الطبيب:\n{answer_text}\n\nحدد أسماء الفئات فقط كما طُلِب أعلاه:"
    messages.append({"role":"user", "content": tgt})
    return messages

def df_to_sft_texts_fewshot_ar_all(df: pd.DataFrame, shots_df: pd.DataFrame, tokenizer):
    texts = []
    for _, row in df.iterrows():
        msgs = build_fewshot_messages_all_ar(row["answer"], shots_df)
        gold = ids_to_ar_string(parse_labels(row["final_AS"]))  # الهدف بالعربية
        msgs = msgs + [{"role":"assistant", "content": gold}]
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False, enable_thinking=False
        )
        texts.append(text)
    return texts

In [ ]:
print(x_col, label_col)

answer final_AS


In [ ]:
def df_to_sft_texts(df: pd.DataFrame):
    texts = []
    for _, row in df.iterrows():
        msgs = build_messages_for_answer(row[x_col])
        gt   = ids_to_ar_string(parse_labels(row[label_col]))   # gold label IDs string
        msgs = msgs + [{"role": "assistant", "content": gt}]
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        texts.append(text)
    return texts



In [ ]:
train_texts = df_to_sft_texts_fewshot_ar_all(train_df, fewshot_sample, tokenizer)
val_texts   = df_to_sft_texts_fewshot_ar_all(val_df, fewshot_sample, tokenizer)

train_ds = Dataset.from_dict({"text": train_texts})
val_ds   = Dataset.from_dict({"text": val_texts})

In [ ]:
print(val_ds[40]['text'])

<bos>أنت مُصنِّف لأساليب ردود الأطباء. مهمتك تحديد الأسلوب/الأساليب التي يندرج تحتها ردُّ الطبيب، وقد تنطبق أكثر من فئة في الوقت نفسه.
الفئات المتاحة:
معلومات (Information): يشمل الإجابات التي تقدّم معلومات أو موارد أو توضيحات، ويشمل أيضاً طلبات الحصول على معلومات.
إرشاد مباشر (Direct Guidance): يشمل الاقتراحات أو التعليمات أو النصائح، ويشمل الإجابات التي تخبر السائل ماذا ينبغي أن يفعل.
دعم عاطفي (Emotional Support): يشمل المواساة أو الطمأنة أو عبارات الدعم والمساندة.

التعليمات: أعد أسماء الفئات بالعربية تمامًا كما كُتبت أعلاه، مفصولة بـ «،» فقط دون أي نص إضافي.<start_of_turn>user
ردّ الطبيب:
يجب أن تتابع مع الطبيب كما يجب أن تتابع مع معالج نفسي متخصص للمساعدة أيضا لحل الأزمات التي تمر بها أو مررت بها وتؤثر على نفسيتك

حدد أسماء الفئات فقط كما طُلِب أعلاه:<end_of_turn>
<start_of_turn>model
معلومات، إرشاد مباشر<end_of_turn>
<start_of_turn>user
ردّ الطبيب:
عليك الذهاب للطبيب لكي يصف العلاج.تناولي اقراص بروزاك

حدد أسماء الفئات فقط كما طُلِب أعلاه:<end_of_turn>
<start_of_turn>model
إرشاد

In [ ]:
# =============== 3) Build datasets ===============
# Assumes you already have: train_df, val_df with columns: 'answer', 'final_AS'
train_texts = df_to_sft_texts(train_df)
val_texts   = df_to_sft_texts(val_df)

train_ds = Dataset.from_dict({"text": train_texts})
val_ds   = Dataset.from_dict({"text": val_texts})

In [ ]:
print(val_ds[40]['text'])

<|im_start|>system
أنت مُصنِّف لأساليب ردود الأطباء. مهمتك تحديد الأسلوب/الأساليب التي يندرج تحتها ردُّ الطبيب، وقد تنطبق أكثر من فئة في الوقت نفسه.
الفئات المتاحة:
معلومات (Information): يشمل الإجابات التي تقدّم معلومات أو موارد أو توضيحات، ويشمل أيضاً طلبات الحصول على معلومات.
إرشاد مباشر (Direct Guidance): يشمل الاقتراحات أو التعليمات أو النصائح، ويشمل الإجابات التي تخبر السائل ماذا ينبغي أن يفعل.
دعم عاطفي (Emotional Support): يشمل المواساة أو الطمأنة أو عبارات الدعم والمساندة.

التعليمات: أعد أسماء الفئات بالعربية تمامًا كما كُتبت أعلاه، مفصولة بـ «،» فقط دون أي نص إضافي.<|im_end|>
<|im_start|>user
ردّ الطبيب:
الحل بسيط يكمن في تحديد موعد مع الطبيب النفسي لمعرفة بعض التفاصل وأخذ السيره المرضيه بشكل مفصل ولتحديد العلاج المناسب لكي لا تترددي في طلب الاستشاره فهي الخلاص الوحيد لما تعانين منه تمنياتي بالشفاء

حدد أسماء الفئات فقط كما طُلِب أعلاه:<|im_end|>
<|im_start|>assistant
<think>

</think>

معلومات، إرشاد مباشر، دعم عاطفي<|im_end|>



In [ ]:
from trl import SFTTrainer  # <- TRL's, not Unsloth's monkey-patched one
from transformers import TrainingArguments
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    # max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=2,
        num_train_epochs=15,
        warmup_steps=5,
        max_steps=60,
        eval_strategy="epoch",  # <-- evaluate each epoch
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=4,
        learning_rate=2e-4,
        # num_train_epochs=15,
        fp16=True,
        bf16=False,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,  # Make sure to set this!
        output_dir="outputs",
        report_to="wandb",
    ),
    # train_on_inputs=False,
)
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,              # <-- validation split
#     dataset_text_field="text",
#     max_seq_length=max_seq_length,
#     packing=False,
#     args=TrainingArguments(
#         output_dir=SAVE_DIR,
#         per_device_train_batch_size=2,
#         per_device_eval_batch_size=2,
#         gradient_accumulation_steps=4,
#         learning_rate=2e-4,
#         num_train_epochs=2,           # change as you like
#         warmup_steps=50,
#         logging_steps=10,
#         evaluation_strategy="epoch",  # <-- evaluate each epoch
#         save_strategy="epoch",
#         load_best_model_at_end=True,
#         report_to="none",
#         seed=SEED,
#         fp16=not is_bfloat16_supported(),
#         bf16=is_bfloat16_supported(),
#         optim="adamw_8bit",
#         weight_decay=0.01,
#         lr_scheduler_type="linear",
#     ),
#     train_on_inputs=False,            # mask user/system tokens from loss
# )


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]:   0%|          | 0/293 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
max_seq_length = 2048   # or whatever you used for tokenization

# make sure both tokenizer and model expose max length
tokenizer.model_max_length = max_seq_length
if not hasattr(model, "max_seq_length"):
    model.max_seq_length = max_seq_length

In [ ]:
trainer.train()

AttributeError: 'Gemma2Model' object has no attribute 'max_seq_length'

In [ ]:
trainer.evaluate()

{'eval_loss': 0.2420462816953659,
 'eval_runtime': 4.6187,
 'eval_samples_per_second': 10.826,
 'eval_steps_per_second': 5.413}

In [ ]:
trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)
wandb: Currently logged in as: mohamad-hassan-rasmy (team258) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.644200,0.550470


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=60, training_loss=0.9416022722919782, metrics={'train_runtime': 126.9332, 'train_samples_per_second': 3.782, 'train_steps_per_second': 0.473, 'total_flos': 7761827224596480.0, 'train_loss': 0.9416022722919782})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.5504697561264038,
 'eval_runtime': 2.6005,
 'eval_samples_per_second': 19.227,
 'eval_steps_per_second': 9.614}

In [ ]:
trainer.evaluate()

{'eval_loss': 0.5509452819824219,
 'eval_runtime': 2.6238,
 'eval_samples_per_second': 19.056,
 'eval_steps_per_second': 9.528}

In [ ]:
import re, torch, pandas as pd
from tqdm import tqdm

# ---------- paths ----------
IN_PATH   = "/content/subtask2_input_test.tsv"   # no header, one column: answer text
OUT_PATH  = "/content/subtask2_with_preds.tsv"

In [ ]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
# ---------- parsing helpers ----------
KNOWN_AR = {"معلومات", "إرشاد مباشر", "دعم عاطفي"}
SEP_RE   = re.compile(r"[,\u060C]+")   # English comma or Arabic comma

def parse_ar_names(gen_text: str):
    # split on commas, trim, keep only exact known names
    parts = [p.strip() for p in SEP_RE.split(gen_text) if p.strip()]
    names = [p for p in parts if p in KNOWN_AR]
    return names

In [ ]:
def names_to_id_string(names):
    ids = sorted({AR2ID[n] for n in names if n in AR2ID})
    return ", ".join(ids)

# ---------- load test file (no header) ----------
df_test = pd.read_csv(IN_PATH, sep="\t", header=None, names=["answer"])
# df_test["answer"] = df_test["answer"].astype(str).fillna("")

# ---------- batched generation ----------
BATCH_SIZE     = 8
MAX_NEW_TOKENS = 16

pred_ar = []

In [ ]:
batch =  df_test.iloc[0:2]["answer"].tolist()

In [ ]:
build_fewshot_messages_all_ar(train_df, fewshot_sample)
train_texts = df_to_sft_texts_fewshot_ar_all(train_df, fewshot_sample, tokenizer)
val_texts   = df_to_sft_texts_fewshot_ar_all(val_df,   fewshot_sample, tokenizer)

train_ds = Dataset.from_dict({"text": train_texts})
val_ds   = Dataset.from_dict({"text": val_texts})

In [ ]:
prompts = [
        tokenizer.apply_chat_template(
            build_fewshot_messages_all_ar(ans, fewshot_sample),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
        for ans in batch
    ]

In [ ]:
print(prompts[0])

<|im_start|>system
أنت مُصنِّف لأساليب ردود الأطباء. مهمتك تحديد الأسلوب/الأساليب التي يندرج تحتها ردُّ الطبيب، وقد تنطبق أكثر من فئة في الوقت نفسه.
الفئات المتاحة:
معلومات (Information): يشمل الإجابات التي تقدّم معلومات أو موارد أو توضيحات، ويشمل أيضاً طلبات الحصول على معلومات.
إرشاد مباشر (Direct Guidance): يشمل الاقتراحات أو التعليمات أو النصائح، ويشمل الإجابات التي تخبر السائل ماذا ينبغي أن يفعل.
دعم عاطفي (Emotional Support): يشمل المواساة أو الطمأنة أو عبارات الدعم والمساندة.

التعليمات: أعد أسماء الفئات بالعربية تمامًا كما كُتبت أعلاه، مفصولة بـ «،» فقط دون أي نص إضافي.<|im_end|>
<|im_start|>user
ردّ الطبيب:
يجب أن تتابع مع الطبيب كما يجب أن تتابع مع معالج نفسي متخصص للمساعدة أيضا لحل الأزمات التي تمر بها أو مررت بها وتؤثر على نفسيتك

حدد أسماء الفئات فقط كما طُلِب أعلاه:<|im_end|>
<|im_start|>assistant
معلومات، إرشاد مباشر<|im_end|>
<|im_start|>user
ردّ الطبيب:
عليك الذهاب للطبيب لكي يصف العلاج.تناولي اقراص بروزاك

حدد أسماء الفئات فقط كما طُلِب أعلاه:<|im_end|>
<|im_start|>ass

In [ ]:
enc = tokenizer(
  [prompts[0]],
  return_tensors="pt",
  # padding=True,
  # truncation=True,
  # max_length=2048
).to(model.device)

with torch.no_grad():
  gen = model.generate(
      **enc,
      max_new_tokens=128,
      do_sample=True,
      use_cache=False,
      temperature=0.7,
      top_p=0.8,
      top_k=20,
      min_p=0.0,
  )


In [ ]:
x = tokenizer.decode(gen[0], skip_special_tokens=True)

In [ ]:
x[-20:]

'\n\nمعلومات، دعم عاطفي'

In [ ]:
pred_ar = []

In [ ]:
for j in range(len(batch)):
    start = enc["input_ids"][j].size(0)
    # For batched generate, we need prompt lengths per item:
    prompt_len = (enc["input_ids"][j] != tokenizer.pad_token_id).sum().item()
    out_tokens = gen[j][prompt_len:]
    text = tokenizer.decode(out_tokens, skip_special_tokens=True).strip()
    names = parse_ar_names(text)
    print(names)
    pred_ar.append("، ".join(names))

['معلومات', 'إرشاد مباشر']
[]


In [ ]:
pred_ar

['معلومات، إرشاد مباشر', '']

In [ ]:
gen

tensor([[151644,   8948,    198,  69682, 124126,  23364,  64604, 125768,  52704,
          73771,  20931,  56794, 125592, 124421,  21360, 130852,  69423, 123877,
         130170,  98719,     13, 141182, 125492, 125857, 124454, 123877, 123993,
         124495,     14,  31382, 125592, 124421,  21360, 128261,  73274,  11798,
         136304, 129375, 124006, 130852,  64604,  73771, 124663, 132261,  68785,
         128835, 129458, 134194, 128773,  63237,  45577, 125165,  77273, 129816,
         131581,    624, 127797, 124082,  47632, 128631, 129215,    510, 132829,
            320,  14873,   1648,  73274, 133498, 124058, 131732,  47632, 128261,
          39434, 124042,  73771,  10176,  23364, 127343, 128264,  23364, 125646,
          13325, 128264,  39434, 124283, 128779,  47632,  68785,  37524, 126464,
         124014, 126899, 124376, 124665,   8532, 126628, 134516, 128248,  23364,
         127343,    624,  91962, 129104,  65398, 127922,    320,  16027,  81561,
           1648,  73274, 133

In [ ]:
# ---------- parsing helpers ----------
# Map Arabic → IDs
AR2ID = {"معلومات": "1", "إرشاد مباشر": "2", "دعم عاطفي": "3"}
KNOWN_AR = set(AR2ID.keys())

# split on English comma or Arabic comma
SEP_RE = re.compile(r"[,\u060C]+")

# Find content **after** the last </think> tag
THINK_CLOSE_RE = re.compile(r"</think>\s*", flags=re.IGNORECASE)

def extract_after_think(s: str) -> str:
    """Return substring after the last </think> (any whitespace allowed)."""
    m = None
    # Use rfind for speed, then trim; fallback to regex if needed
    pos = s.lower().rfind("</think>")
    if pos != -1:
        return s[pos + len("</think>"):].lstrip()
    # fallback (should rarely hit)
    m = THINK_CLOSE_RE.search(s)
    return s[m.end():].lstrip() if m else s.strip()

def parse_ar_names(text: str):
    """Keep only exact known Arabic labels."""
    parts = [p.strip() for p in SEP_RE.split(text) if p.strip()]
    return [p for p in parts if p in KNOWN_AR]

def names_to_id_line(names):
    """['معلومات','دعم عاطفي'] -> '1, 3' (sorted numerically)."""
    ids = sorted({AR2ID[n] for n in names if n in AR2ID}, key=int)
    return ", ".join(ids)

# ---------- load test file (no header) ----------
df_test = pd.read_csv(IN_PATH, sep="\t", header=None, names=["answer"])
df_test["answer"] = df_test["answer"].astype(str).fillna("")

# ---------- run inference ----------
BATCH_SIZE     = 2
MAX_NEW_TOKENS = 18



In [ ]:
pred_id_lines = []
BATCH_SIZE = 1
for i in tqdm(range(0, len(df_test), BATCH_SIZE)):
    batch = df_test.iloc[i]["answer"]
    # print(batch)
    # break

    # build prompts with chat template
    prompts = tokenizer.apply_chat_template(
            build_fewshot_messages_all_ar(batch, fewshot_sample),
            # build_messages_for_answer(batch),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

    # print(type(prompts))
    # break
    enc = tokenizer(
        [prompts],
        return_tensors="pt",
        padding=True,
        truncation=True,
        # max_length=2048
    ).to(model.device)

    with torch.no_grad():
        gen = model.generate(
            **enc,
            max_new_tokens=128,
            do_sample=False,   # Deterministic outputs for accuracy
            temperature=0.0,   # Ignored if do_sample=False
            use_cache=True
            # do_sample=True,
            # use_cache=False,
            # temperature=0.7,
            # top_p=0.8,
            # top_k=20,
            # min_p=0.0,
        )

        #     max_new_tokens=MAX_NEW_TOKENS,
        #     do_sample=False,
        #     # pad_token_id=tokenizer.eos_token_id
        # )


    # Decode **full sequence** (no manual prompt slicing); then take the text after </think>
    full_txts = tokenizer.decode(gen[0], skip_special_tokens=True)
    # print(full_txts)
    # break

    # append the full_txts to pred_id_lines
    # for full in full_txts:
    pred_id_lines.append(full_txts)

    # for full in full_txts:
    #     tail = extract_after_think(full)        # everything after </think>
    #     names = parse_ar_names(tail)            # Arabic categories present
    #     pred_id_lines.append(names_to_id_line(names))   # "1, 3" or "" if none

100%|██████████| 150/150 [01:04<00:00,  2.31it/s]


In [ ]:
# pred_id_lines = []

In [ ]:
len(pred_id_lines)

150

In [ ]:
print(pred_id_lines[7][-50:])

ِب أعلاه:
assistant
<think>

</think>

إرشاد مباشر


In [ ]:
print(pred_id_lines[7])

system
أنت مُصنِّف لأساليب ردود الأطباء. مهمتك تحديد الأسلوب/الأساليب التي يندرج تحتها ردُّ الطبيب، وقد تنطبق أكثر من فئة في الوقت نفسه.
الفئات المتاحة:
معلومات (Information): يشمل الإجابات التي تقدّم معلومات أو موارد أو توضيحات، ويشمل أيضاً طلبات الحصول على معلومات.
إرشاد مباشر (Direct Guidance): يشمل الاقتراحات أو التعليمات أو النصائح، ويشمل الإجابات التي تخبر السائل ماذا ينبغي أن يفعل.
دعم عاطفي (Emotional Support): يشمل المواساة أو الطمأنة أو عبارات الدعم والمساندة.

التعليمات: أعد أسماء الفئات بالعربية تمامًا كما كُتبت أعلاه، مفصولة بـ «،» فقط دون أي نص إضافي.
user
ردّ الطبيب:
عليك الالتزام بالمتابعة والعلاج الدوائي والنفسي

حدد أسماء الفئات فقط كما طُلِب أعلاه:
assistant
<think>

</think>

إرشاد مباشر


In [ ]:
preds = []
for full in pred_id_lines:
    tail = extract_after_think(full)        # everything after </think>
    names = parse_ar_names(tail)            # Arabic categories present
    preds.append(names_to_id_line(names))   # "1, 3" or "" if none

In [ ]:
# fewshot
preds

['1, 3',
 '1, 2',
 '1',
 '1, 3',
 '1, 2',
 '1, 2',
 '1',
 '2',
 '1',
 '1',
 '1, 3',
 '1, 2',
 '1',
 '1',
 '1',
 '2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1',
 '1',
 '1',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1',
 '1, 2',
 '1',
 '1, 3',
 '1',
 '1',
 '1',
 '1, 2',
 '1, 2',
 '1',
 '1, 3',
 '2',
 '1',
 '1',
 '1',
 '2',
 '1',
 '3',
 '1',
 '1',
 '1',
 '1',
 '1, 2',
 '1',
 '1, 2',
 '1',
 '1',
 '1',
 '1, 3',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1',
 '2',
 '1, 2',
 '1',
 '1',
 '1, 3',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1',
 '2',
 '1, 2',
 '2',
 '1',
 '1, 2',
 '1',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1',
 '1',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '2',
 '1, 2',
 '2',
 '1',
 '1',
 '1',
 '1',
 '1, 2',
 '2',
 '1, 2',
 '1',
 '2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1',
 '1, 2',
 '1',
 '1, 2',
 '1, 2',
 '1, 2',
 '1',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 3',
 '1',
 '1, 2',
 '1, 2',
 

In [ ]:
preds

['1, 2',
 '1',
 '1',
 '1',
 '1, 2',
 '1',
 '1',
 '2',
 '1',
 '1, 2',
 '1',
 '1, 2, 3',
 '1',
 '2',
 '1',
 '2',
 '1',
 '1',
 '1, 2',
 '2',
 '1, 2',
 '1',
 '1',
 '1',
 '1, 2',
 '1',
 '1',
 '1, 2',
 '1, 2',
 '1, 2',
 '1',
 '2',
 '1',
 '1',
 '1',
 '2',
 '1',
 '1, 2',
 '1',
 '1',
 '1, 2, 3',
 '1, 2',
 '1, 2',
 '1',
 '1',
 '1, 2',
 '1, 2',
 '1',
 '1',
 '2',
 '1',
 '1',
 '1, 2',
 '1',
 '1',
 '1, 2',
 '1',
 '1',
 '1',
 '1',
 '1, 2, 3',
 '1, 2',
 '2',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '2',
 '2',
 '1',
 '1, 2',
 '1',
 '1, 2',
 '1',
 '1',
 '1',
 '2',
 '1',
 '1',
 '1',
 '1',
 '2',
 '1, 2',
 '1, 2',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1, 2',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1, 2',
 '1',
 '1, 2',
 '1',
 '1',
 '1',
 '1',
 '2',
 '1',
 '1, 2',
 '1',
 '1',
 '1, 2',
 '1',
 '2',
 '1',
 '2',
 '1',
 '1',
 '1, 2',
 '2',
 '1',
 '1, 2',
 '1',
 '1, 2, 3',
 '1',
 '1',
 '1',
 '1, 2',
 '1, 2',
 '1',
 '2',
 '1',
 '2',
 '1, 2',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1,

In [ ]:
import re, numpy as np, pandas as pd
from sklearn.metrics import f1_score, jaccard_score

# ==== paths ====
GT_PATH = "/content/subtask2_output_test.tsv"   # adjust if needed

# ==== 1) load GT (one label-combination per line like: "2, 3", "1") ====
gt_series = pd.read_csv(GT_PATH, header=None, names=["raw"], sep="\t", engine="python")["raw"].astype(str).fillna("")

# robust parser: picks 1/2/3 anywhere in the line
def parse_ids_line(line: str):
    return re.findall(r"[123]", line)

gt_lists = [parse_ids_line(s) for s in gt_series.tolist()]
pred_lists = [parse_ids_line(s) for s in preds]   # ← from your inference

# align lengths defensively
n = min(len(gt_lists), len(pred_lists))
if len(gt_lists) != len(pred_lists):
    print(f"⚠️ Length mismatch: GT={len(gt_lists)} vs PRED={len(pred_lists)} → evaluating on first {n} rows.")
gt_lists   = gt_lists[:n]
pred_lists = pred_lists[:n]

In [ ]:
n

150

In [ ]:
# ==== 2) convert to binary matrices (N x 3) ====
def to_bin(list_of_lists, num_labels=3):
    arr = np.zeros((len(list_of_lists), num_labels), dtype=int)
    for i, labs in enumerate(list_of_lists):
        for lab in labs:
            idx = int(lab) - 1
            if 0 <= idx < num_labels:
                arr[i, idx] = 1
    return arr

gt_bin   = to_bin(gt_lists,   3)
pred_bin = to_bin(pred_lists, 3)

In [ ]:
# fewshot
pred_bin

array([[1, 0, 1],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 1],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 1],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 1],
       [0, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [0, 0, 1],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1,

In [ ]:
pred_bin

array([[1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 1],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [0, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1,

In [ ]:
# fewshot


# ==== 3) metrics ====
f1_weighted  = f1_score(gt_bin, pred_bin, average="weighted", zero_division=0)
jacc_samples = jaccard_score(gt_bin, pred_bin, average="samples", zero_division=0)

print({"f1_weighted": f1_weighted, "jaccard_samples": jacc_samples})


{'f1_weighted': 0.7379255760279514, 'jaccard_samples': np.float64(0.6511111111111111)}


In [ ]:



# ==== 3) metrics ====
f1_weighted  = f1_score(gt_bin, pred_bin, average="weighted", zero_division=0)
jacc_samples = jaccard_score(gt_bin, pred_bin, average="samples", zero_division=0)

print({"f1_weighted": f1_weighted, "jaccard_samples": jacc_samples})


{'f1_weighted': 0.6460119953484513, 'jaccard_samples': np.float64(0.5888888888888889)}


In [ ]:
pred_id_lines[1]

''

In [ ]:
len(pred_ar)

150

In [ ]:


# ---------- use your trainer’s model & tokenizer ----------
model = trainer.model
tokenizer = trainer.tokenizer if hasattr(trainer, "tokenizer") else tokenizer
model.eval()




# (optional) map Arabic → IDs later if you want numeric labels
# AR2ID = {"معلومات":"1", "إرشاد مباشر":"2", "دعم عاطفي":"3"}

def names_to_id_string(names):
    ids = sorted({AR2ID[n] for n in names if n in AR2ID})
    return ", ".join(ids)

# ---------- load test file (no header) ----------
df_test = pd.read_csv(IN_PATH, sep="\t", header=None, names=["answer"])
df_test["answer"] = df_test["answer"].astype(str).fillna("")

# ---------- batched generation ----------
BATCH_SIZE     = 8
MAX_NEW_TOKENS = 12

pred_ar = []

for i in tqdm(range(0, len(df_test), BATCH_SIZE)):
    batch = df_test.iloc[i:i+BATCH_SIZE]["answer"].tolist()

    # build prompts with chat template
    prompts = [
        tokenizer.apply_chat_template(
            build_messages_for_answer(ans),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
        for ans in batch
    ]

    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        gen = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # slice off the prompt and decode only the generated tail
    for j in range(len(batch)):
        start = enc["input_ids"][j].size(0)
        # For batched generate, we need prompt lengths per item:
        prompt_len = (enc["input_ids"][j] != tokenizer.pad_token_id).sum().item()
        out_tokens = gen[j][prompt_len:]
        text = tokenizer.decode(out_tokens, skip_special_tokens=True).strip()
        names = parse_ar_names(text)
        pred_ar.append("، ".join(names))

ValueError: No columns in the dataset match the model's forward method signature: (input_ids, labels, seq_lengths, completion_mask, assistant_masks). The following columns have been ignored: [text]. Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`.

In [ ]:

def df_to_sft_texts(df: pd.DataFrame):
    texts = []
    for _, row in df.iterrows():
        msgs = build_messages_for_answer(row["answer"])
        gt   = labels_to_id_string(parse_labels(row["final_AS"]))   # gold label IDs string
        msgs = msgs + [{"role": "assistant", "content": gt}]
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        texts.append(text)
    return texts

In [ ]:
import ast, re, random
from typing import List, Dict

LABEL_DESC = {"1": "Information", "2": "Direct Guidance", "3": "Emotional Support"}
FEWSHOT_K  = 3  # how many exemplars per sample

def parse_labels(cell) -> List[str]:
    if isinstance(cell, list): return [str(x) for x in cell]
    if isinstance(cell, str):
        try:
            obj = ast.literal_eval(cell)
            if isinstance(obj, (list, tuple)): return [str(x) for x in obj]
        except Exception:
            pass
        return re.findall(r"[123]", cell)
    return []


ID2AR = {"1": "معلومات", "2": "إرشاد مباشر", "3": "دعم عاطفي"}
AR2ID = {v: k for k, v in ID2AR.items()}

def ids_to_ar_string(ids):
    """['1','3'] → 'معلومات، دعم عاطفي' (Arabic comma)."""
    ordered = [ID2AR[i] for i in sorted({str(i) for i in ids}) if i in ID2AR]
    return "، ".join(ordered)


def labels_to_id_string(labels: List[str]) -> str:
    # We’ll train the model to emit "1, 3" (not names). Easier to parse later.
    labs = sorted({str(l) for l in labels})
    return ", ".join(labs)


def build_messages_for_answer(answer: str, shots_df) -> List[Dict[str, str]]:
    """
    Few-shot chat messages for ONE training example.
    We show K exemplars (answer → labels), then ask for the target answer.
    """
    sys = (
      "You classify an input ANSWER into one or more response styles.\n"
      "Available labels: 1=Information, 2=Direct Guidance, 3=Emotional Support.\n"
      "Return a comma-separated list of label IDs only (e.g., 1, 3)."
    )
    messages = [{"role": "system", "content": sys}]

    exemplars = shots_df.sample(n=min(FEWSHOT_K, len(shots_df)), random_state=random.randint(0, 10**6))
    for _, r in exemplars.iterrows():
        ex_labels = labels_to_id_string(parse_labels(r["final_AS"]))
        ex_user   = f"ANSWER:\n{r['answer']}\n\nRequired label IDs:"
        messages.append({"role": "user", "content": ex_user})
        messages.append({"role": "assistant", "content": ex_labels})

    # target query (no answer yet)
    tgt_user = f"ANSWER:\n{answer}\n\nRequired label IDs:"
    messages.append({"role": "user", "content": tgt_user})
    return messages


In [ ]:
def df_to_sft_texts(df, shots_df, tokenizer):
    texts = []
    for _, row in df.iterrows():
        msgs = build_messages_for_answer(row["answer"], shots_df)
        # For TRAIN we include the assistant ground truth so SFT has a target.
        gt = labels_to_id_string(parse_labels(row["final_AS"]))
        msgs = msgs + [{"role": "assistant", "content": gt}]
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False, enable_thinking=False
        )
        texts.append(text)
    return texts


In [ ]:
# --- build SFT texts from your data (answers!) ---
train_texts = df_to_sft_texts(train_df, fewshot_sample, tokenizer)
val_texts   = df_to_sft_texts(val_df,   fewshot_sample, tokenizer)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_dict({"text": train_texts})
val_ds   = Dataset.from_dict({"text": val_texts})

In [ ]:
print(train_ds['text'][0])

<|im_start|>system
You classify an input ANSWER into one or more response styles.
Available labels: 1=Information, 2=Direct Guidance, 3=Emotional Support.
Return a comma-separated list of label IDs only (e.g., 1, 3).<|im_end|>
<|im_start|>user
ANSWER:
يجب أن تتابع مع الطبيب كما يجب أن تتابع مع معالج نفسي متخصص للمساعدة أيضا لحل الأزمات التي تمر بها أو مررت بها وتؤثر على نفسيتك

Required label IDs:<|im_end|>
<|im_start|>assistant
1, 2<|im_end|>
<|im_start|>user
ANSWER:
لاتكترثي لذلك لست أنت من صنعت شكلك, لكن هناك أمور يمكنك تحسينها فموضو الأسنان بالتقويم لدى أخصائي الأسنان تعود وتكسبك جمالا

Required label IDs:<|im_end|>
<|im_start|>assistant
3<|im_end|>
<|im_start|>user
ANSWER:
اهلا عزيزي  اقترح عليك ان تذهب الى طبيب نفسي ليصرف لك العلاج المناسب  لا تقلق مشكلتك وارده لكل شخص فقط التزم بالعلاج المصرف من قبل الطبيب وستكون بخير باذن الله  انتظر منك ان تطمئني بعدها على صحتك

Required label IDs:<|im_end|>
<|im_start|>assistant
2, 3<|im_end|>
<|im_start|>user
ANSWER:
زيادة الضغوط المرحليه في

In [ ]:
# --- Prompt Preparation Functions ---
def make_prompt(instruction):
    return [{"role": "user", "content": instruction}]


def apply_chat_template(prompt, tokenizer, enable_thinking=True):
    messages = make_prompt(prompt)
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )


@weave.op
def generate_response(prompt, enable_thinking=True):
    prompt_text = apply_chat_template(prompt, TOKENIZER, enable_thinking)
    inputs = TOKENIZER([prompt_text], return_tensors="pt").to("cuda")
    with torch.no_grad():
        gen_output = BASE_MODEL.generate(
            **inputs,
            max_new_tokens=32,
            use_cache=False,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            min_p=0.0,
        )
    output_text = TOKENIZER.decode(gen_output[0], skip_special_tokens=True)
    return output_text


In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(
    MODEL_ID,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    force_download=True,
)
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, config=cfg).cuda()


config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

In [ ]:
from transformers import BitsAndBytesConfig
from peft import get_peft_model, LoraConfig
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID,
            config=cfg,
            quantization_config=bnb_cfg,
            force_download=True,
            trust_remote_code=True
        ).to('cuda')


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some weights of Gemma2ForSequenceClassification were not initialized from the model checkpoint at silma-ai/SILMA-9B-Instruct-v1.0 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model

Gemma2ForSequenceClassification(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=3584, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=3584, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((3584,), eps=1e-

In [ ]:
# 1⃣  free anything left from earlier failed loads
import gc, torch
gc.collect(); torch.cuda.empty_cache()

# 2⃣  tell PyTorch to use expandable segments (helps fragmentation)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3⃣  gradient checkpointing before LoRA adapters
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True
        )
model.config.use_cache = False          # speeds up checkpointing


# 3️⃣ attach trainable LoRA adapters
# lora_cfg = LoraConfig(
#     r=8, lora_alpha=16, lora_dropout=0.05,
#     target_modules=["q_proj", "v_proj"],   # Qwen uses these names
#     bias="none",
#     task_type="SEQ_CLS"                    # anything except CAUSAL_LM works here
# )
lora_cfg = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="SEQ_CLS",
)

model = get_peft_model(model, lora_cfg)

In [ ]:
model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Gemma2ForSequenceClassification(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 3584, padding_idx=0)
        (layers): ModuleList(
          (0-41): 42 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDic

In [ ]:
model.config.eos_token_id

1

In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
def binrow_to_labels(row):
    return [id2label[i] for i, v in enumerate(row) if v == 1]

In [ ]:
def to_multihot(labels):
    y = [0] * NUM_LABELS
    for lab in labels:
        if lab in label2id:
            y[label2id[lab]] = 1
    return y

def make_hf_dataset(df):
    texts, labels = [], []
    for _, row in df.iterrows():
        labs = parse_label_list(row[label_col])
        texts.append(row[x_col])      # no prompting; raw text only
        labels.append([float(x) for x in to_multihot(labs)])
    return Dataset.from_dict({"text": texts, "labels": labels})

In [ ]:
train_ds = make_hf_dataset(train_df)
val_ds   = make_hf_dataset(val_df)

In [ ]:
train_ds[1]

{'text': 'من المفيد مراجعة متخصص للتشخيص الدقيق والعلاج',
 'labels': [0.0, 1.0, 0.0]}

In [ ]:
def tok_fn(batch):
    return tokenizer(batch["text"],
                     truncation=True,
                     add_special_tokens=True)

train_ds = train_ds.map(tok_fn, batched=True, remove_columns=["text"])
val_ds   = val_ds.map(tok_fn,   batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
# ----------------------------------------------------------
# 4.  METRICS
# ----------------------------------------------------------
from scipy.special import expit

def compute_metrics(eval_pred):
    logits, gold = eval_pred
    probs = expit(logits)       # sigmoid
    preds = (probs >= 0.5).astype(int)

    f1  = f1_score(gold, preds, average="weighted", zero_division=0)
    jac = jaccard_score(gold, preds, average="samples", zero_division=0)
    return {"weighted_f1": f1, "jaccard": jac}

In [ ]:
from transformers.optimization import get_constant_schedule

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)  # Adam as requested
scheduler = get_constant_schedule(optimizer)                # constant LR (no warmup, no decay)

In [ ]:
from transformers.optimization import get_constant_schedule

# ============= training args (your hypers) =============
# epochs=15, batch=8, lr=2e-5, optimizer=Adam, early-stop-patience=10
training_args = TrainingArguments(
    output_dir="./marbert_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=4,
    # lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="weighted_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED
)

# ------- Build Adam (NOT AdamW) + linear scheduler --------

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)  # Adam as requested
scheduler = get_constant_schedule(optimizer)                # constant LR (no warmup, no decay)

# ============= trainer =============
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)]
)


/tmp/ipython-input-4229318525.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
!rm -rf /content/marbert_results

In [ ]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,No log,0.628502,0.734165,0.623333
2,0.900900,0.582692,0.769198,0.670000
3,0.204900,1.041903,0.631052,0.583333
4,0.090400,1.002190,0.765781,0.673333
5,0.090400,1.093850,0.746634,0.683333
6,0.030900,1.278895,0.780618,0.700000
7,0.019200,1.625516,0.766493,0.676667
8,0.023900,1.313863,0.770732,0.690000
9,0.023900,1.601186,0.744745,0.666667
10,0.005800,1.522735,0.753724,0.676667


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/

TrainOutput(global_step=570, training_loss=0.11427774541733558, metrics={'train_runtime': 930.5007, 'train_samples_per_second': 4.836, 'train_steps_per_second': 0.613, 'total_flos': 4.379982874681344e+16, 'train_loss': 0.11427774541733558, 'epoch': 15.0})

In [ ]:
# Optional: final eval after training completes
final_metrics = trainer.evaluate()
print("Final eval:", final_metrics)

Final eval: {'eval_loss': 1.278895378112793, 'eval_weighted_f1': 0.7806184012066365, 'eval_jaccard': 0.7, 'eval_runtime': 3.0098, 'eval_samples_per_second': 16.612, 'eval_steps_per_second': 2.326, 'epoch': 15.0}


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset

# Path to input & output
INPUT_PATH  = "/content/subtask2_input_test.tsv"   # adjust if needed
OUTPUT_PATH = "/content/subtask2_pred_output.tsv" # your final file

if task == 1:
    INPUT_PATH  = "/content/subtask1_input_test.tsv"   # adjust if needed
    OUTPUT_PATH = "/content/subtask1_pred_output.tsv" # your final file
elif task == 3:
    INPUT_PATH  = "/content/subtask3_input_test.tsv"   # adjust if needed
    OUTPUT_PATH = "/content/subtask3_pred_output.tsv" # your final file

# 1) Load the test TSV (no header)
df_test = pd.read_csv(INPUT_PATH, sep="\t", header=None, names=["text"])


In [ ]:
# 2) Tokenize (no prompting; raw text)
test_ds = Dataset.from_dict({"text": df_test["text"].tolist()})
test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"])


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [ ]:
test_ds

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 150
})

In [ ]:
len(test_ds[3]['input_ids'])

51

In [ ]:

# 3) Predict with your trained model
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
probs  = expit(logits)
pred_bin = (probs >= 0.5).astype(int)

# find when pred_bin has [0,0,0]

In [ ]:
# find when pred_bin has [0,0,0]
# find when pred_bin has [0,0,0]
zero_rows_indices = [i for i, row in enumerate(pred_bin) if np.sum(row) == 0]

print("Indices of rows with all zeros:", zero_rows_indices)

Indices of rows with all zeros: [79]


In [ ]:
probs[3]

array([0.99171644, 0.6774737 , 0.00466371], dtype=float32)

In [ ]:
df_test.loc[3].values[0]

'مساء الخير اجعل لنفسك بجانب العبادات انشطه اخرى مثل ممارسة رياضه خفيفه والقراءه في كتب بتحبها ووجودك مع الاصدقاء ده هيساعدك كنير'

In [ ]:
pred_bin

array([[1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 1],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [0, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [0, 1, 0],
       [1, 1, 1],
       [0, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1,

In [ ]:
len(pred_bin)

150

In [ ]:
def binrow_to_labels(row):
    return ", ".join(id2label[i] for i, v in enumerate(row) if v == 1)

pred_lines = [binrow_to_labels(row) for row in pred_bin]  # e.g. "A, B, D"

In [ ]:
type(pred_lines[0])

str

In [ ]:
# pd.Series(pred_lines).to_csv(OUTPUT_PATH, index=False, header=False)

print(f"✅ Saved predictions in required format to {OUTPUT_PATH}")

✅ Saved predictions in required format to /content/subtask2_pred_output.tsv


In [ ]:
import pandas as pd
import csv

def binrow_to_labels(row):
    return ", ".join(id2label[i] for i, v in enumerate(row) if v == 1)

pred_lines = [binrow_to_labels(row) for row in pred_bin]

pd.Series(pred_lines).to_csv(
    OUTPUT_PATH,
    sep="\t",
    index=False,
    header=False,
    quoting=csv.QUOTE_MINIMAL,   # or just omit 'quoting' entirely
)
print(f"✅ Saved {len(pred_lines)} predictions to {OUTPUT_PATH}")

✅ Saved 150 predictions to /content/subtask2_pred_output.tsv


In [ ]:
# 4) Convert binary predictions → label letters
# Assumes you already have `id2label` from training
def binrow_to_labels(row):
    return ", ".join(id2label[i] for i, v in enumerate(row) if v == 1)

pred_lines = [binrow_to_labels(row) for row in pred_bin]

# 5) Save as single-column TSV, no header
# pd.Series(pred_lines).to_csv(OUTPUT_PATH, index=False, header=False)
import csv


pd.Series(pred_lines).to_csv(
    OUTPUT_PATH,
    sep="\t",
    index=False,
    header=False,
    quoting=csv.QUOTE_NONE,
    escapechar="\\",   # required when QUOTE_NONE
)


print(f"✅ Saved predictions in required format to {OUTPUT_PATH}")


Error: single empty field record must be quoted

In [ ]:
def binrow_to_strlist(row):
    ids = [str(i+1) for i, v in enumerate(row) if v == 1]
    return "[" + ", ".join(f"'{x}'" for x in ids) + "]"



In [ ]:
train_ds

In [ ]:
# ------------- helpers -------------
def parse_labels(cell):
    """
    Robustly parse labels from various formats:
    - "['1','3']"   -> ['1','3']
    - "[1, 3]"      -> ['1','3']
    - "1, 3"        -> ['1','3']
    - ["1","2"]     -> ['1','2']
    """
    if isinstance(cell, list):
        return [str(x) for x in cell]
    if isinstance(cell, str):
        try:
            obj = ast.literal_eval(cell)
            if isinstance(obj, (list, tuple)):
                return [str(x) for x in obj]
        except Exception:
            pass
        return re.findall(r"[123]", cell)
    # fallback
    return []

def to_multihot(label_list, num_labels=NUM_LABELS):
    y = [0]*num_labels
    for lab in label_list:
        idx = int(lab) - 1
        if 0 <= idx < num_labels:
            y[idx] = 1
    return y

In [ ]:
# ------------- helpers -------------
def parse_labels(cell):
    """
    Robustly parse labels from various formats:
    - "['1','3']"   -> ['1','3']
    - "[1, 3]"      -> ['1','3']
    - "1, 3"        -> ['1','3']
    - ["1","2"]     -> ['1','2']
    """
    if isinstance(cell, list):
        return [str(x) for x in cell]
    if isinstance(cell, str):
        try:
            obj = ast.literal_eval(cell)
            if isinstance(obj, (list, tuple)):
                return [str(x) for x in obj]
        except Exception:
            pass
        return re.findall(r"[123]", cell)
    # fallback
    return []

def to_multihot(label_list, num_labels=NUM_LABELS):
    y = [0]*num_labels
    for lab in label_list:
        idx = int(lab) - 1
        if 0 <= idx < num_labels:
            y[idx] = 1
    return y



def make_hf_dataset(df):
    texts, labels = [], []
    for _, row in df.iterrows():
        labs = parse_labels(row["final_AS"])
        texts.append(row["question"])
        labels.append(to_multihot(labs))
    return Dataset.from_dict({"text": texts, "labels": labels})

train_ds = make_hf_dataset(train_df)
val_ds   = make_hf_dataset(val_df)

# ============= tokenizer / model =============
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
cfg = AutoConfig.from_pretrained(
    MODEL_ID,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    hidden_dropout_prob=0.1,          # dropout = 0.1
    attention_probs_dropout_prob=0.1
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, config=cfg)

def tok_fn(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
    return enc

train_ds = train_ds.map(tok_fn, batched=True, remove_columns=["text"])
val_ds   = val_ds.map(tok_fn,   batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")

# ============= metrics =============
def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_true = np.array(y_true)
    probs = 1 / (1 + np.exp(-logits))
    y_pred = (probs >= 0.5).astype(int)

    f1_w   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    jac_s  = jaccard_score(y_true, y_pred, average="samples", zero_division=0)
    return {"f1_weighted": f1_w, "jaccard_samples": jac_s}

# ============= training args (your hypers) =============
# epochs=15, batch=8, lr=2e-5, optimizer=Adam, early-stop-patience=10
training_args = TrainingArguments(
    output_dir="./marbert_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    report_to="none",
    seed=SEED
)

# ------- Build Adam (NOT AdamW) + linear scheduler --------
total_steps = (len(train_ds) // training_args.per_device_train_batch_size) * training_args.num_train_epochs
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)  # Adam
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.06 * total_steps),   # small warmup (6%)
    num_training_steps=total_steps
)

# ============= trainer =============
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)]
)

trainer.train()

# Optional: final eval after training completes
final_metrics = trainer.evaluate()
print("Final eval:", final_metrics)


In [ ]:
def make_hf_dataset(df, shot_df):
    """
    Build an HF Dataset with two columns:
      • text   : the full few‑shot prompt (str)
      • labels : multi‑hot vector *as float32* (list[float])
    """
    texts, labels = [], []

    for _, row in df.iterrows():
        # ---- 1. labels → multi‑hot → float --------------------
        raw = row["final_AS"]
        raw = ast.literal_eval(raw) if isinstance(raw, str) else raw
        labels.append([float(x) for x in multihot(raw)])   # e.g. [0., 1., 0.]

        # ---- 2. prompt built from the *question* ---------------
        texts.append(build_prompt(row["question"], shot_df))

    return Dataset.from_dict({"text": texts, "labels": labels})


In [ ]:
train_dataset = make_hf_dataset(train_df, fewshot_sample)
val_dataset   = make_hf_dataset(val_df,   fewshot_sample)

# train_dataset = train_dataset.map(tok_fn, batched=True, remove_columns=["text"])
# val_dataset   = val_dataset.map(tok_fn,   batched=True, remove_columns=["text"])

In [ ]:
print(train_dataset['text'][0])

You are an assistant that determines the appropriate response style(s) for a given user query in a mental health context. A query may require one or more of the following styles:
- Information: Provide factual information, resources, or requests for information.
- Direct Guidance: Offer suggestions, instructions, or advice, including actions the user should take to address their situation.
- Emotional Support: Provide approval, reassurance, empathy, or other forms of non-informational emotional support.

Classify the input into all applicable styles based on its content.

### Example
User's Query: "ودي استفسر بخصوص بروزاك 20 كتبه لي اخصائي نفسي كنت استخدم بروكسات 20 و10 ووقفته انا من نفسي لانه سبب كسل ونوم ووزن زايذ راجعت دكتور نفسي اعطاني بروزاك 20 لكن لم استخدمه"
Needed Answer Style(s): Information, Direct Guidance

### Example
User's Query: "ماهو افضل علاج دوائي للاكتئاب وماهي اعراضه الجانبيه علما اني لا استطيع الذهاب لطبيب نفسي للعلاج"
Needed Answer Style(s): Direct Guidance

### E

In [ ]:
# ----------------------------------------------------------
# 2.  DATA
# ----------------------------------------------------------
# ▸ ASSUMPTION: train_df, val_df, fewshot_sample already exist
train_dataset = make_hf_dataset(train_df, fewshot_sample)
val_dataset   = make_hf_dataset(val_df,   fewshot_sample)

def tok_fn(batch):
    return tokenizer(batch["text"],
                     truncation=True,
                     add_special_tokens=True)

train_dataset = train_dataset.map(tok_fn, batched=True, remove_columns=["text"])
val_dataset   = val_dataset.map(tok_fn,   batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")


Map:   0%|          | 0/274 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

In [ ]:
# ---------------------------------------------
# 0.  environment (do this in the very first cell / script line)
# ---------------------------------------------
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------------------------------------------
# 1.  load tokenizer & model with flash-attn
# ---------------------------------------------
from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
import torch, ast, random, numpy as np, pandas as pd

MODEL_ID  = "Qwen/Qwen3-8B"
CTX_MAX   = 2048                 # cap at 4 k; trim only those few above it

# bnb_cfg = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16
# )

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.model_max_length = CTX_MAX     # we will truncate only if >4 k

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_cfg,
#     device_map="auto",
#     trust_remote_code=True,
#     use_flash_attn=True                  # <<< cut attention memory in half
# )

# # prep for k-bit LoRA
# model = prepare_model_for_kbit_training(model)
# model.gradient_checkpointing_enable()    # <<< saves ~40 % activation mem

# lora_cfg = LoraConfig(
#     r=16, lora_alpha=32, lora_dropout=0.05,
#     target_modules=["q_proj", "v_proj"],
#     bias="none", task_type="CAUSAL_LM"
# )
# model = get_peft_model(model, lora_cfg)

# # ---------------------------------------------
# # 2.  tokenizer call that truncates ONLY >CTX_MAX
# # ---------------------------------------------
# def tok_fn(batch):
#     return tokenizer(batch["text"],
#                      truncation=True)     # only sequences above 4 k are cut


In [ ]:
# ----------------------------------------------------------
# 3.  MODEL
# ----------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL,
            num_labels=len(LABEL_ID),
            problem_type="multi_label_classification",
            # quantization_config=bnb_cfg,
            trust_remote_code=True
        ).to(DEVICE)


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model

Qwen3ForSequenceClassification(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attentio

In [ ]:
# 4️⃣ Freeze all parameters except the classification head `score`
for name, param in model.named_parameters():
    param.requires_grad = ("score" in name)  # only score layer is trainable

In [ ]:
# 1⃣  free anything left from earlier failed loads
import gc, torch
gc.collect(); torch.cuda.empty_cache()

# 2⃣  tell PyTorch to use expandable segments (helps fragmentation)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3⃣  gradient checkpointing before LoRA adapters
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True
        )
model.config.use_cache = False          # speeds up checkpointing


In [ ]:
# 3️⃣ attach trainable LoRA adapters
lora_cfg = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   # Qwen uses these names
    bias="none",
    task_type="SEQ_CLS"                    # anything except CAUSAL_LM works here
)
model = get_peft_model(model, lora_cfg)

In [ ]:
model

Qwen3ForSequenceClassification(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear4bit(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1

In [ ]:
# ----------------------------------------------------------
# 4.  METRICS
# ----------------------------------------------------------
from scipy.special import expit

def compute_metrics(eval_pred):
    logits, gold = eval_pred
    probs = expit(logits)       # sigmoid
    preds = (probs >= 0.5).astype(int)

    f1  = f1_score(gold, preds, average="weighted", zero_division=0)
    jac = jaccard_score(gold, preds, average="samples", zero_division=0)
    return {"weighted_f1": f1, "jaccard": jac}

In [ ]:
# ----------------------------------------------------------
# 5.  TRAIN ARGS
# ----------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=8,     # Qwen-8B is heavy → keep small
    per_device_eval_batch_size=8,
    # learning_rate=2e-5,
    # weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="epoch",
    seed=SEED,
    # save_total_limit=1,
    # shrink checkpoint size
    # # save_only_model=True,              # saves model weights only (no optimizer/scheduler)
    # # save_safetensors=False,
    # # load_best_model_at_end=True,
    # metric_for_best_model="weighted_f1",
    # greater_is_better=True,
    report_to="none"
)


In [ ]:
# ----------------------------------------------------------
# 6.  TRAINER
# ----------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

/tmp/ipython-input-3996475930.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,0.775600,0.573230,0.702926,0.606280
2,0.540900,0.582850,0.710008,0.618357


TrainOutput(global_step=70, training_loss=0.65821533203125, metrics={'train_runtime': 554.7649, 'train_samples_per_second': 0.988, 'train_steps_per_second': 0.126, 'total_flos': 2.3100632433106944e+16, 'train_loss': 0.65821533203125, 'epoch': 2.0})

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,0.524500,0.576014,0.727832,0.644928
2,0.522800,0.576971,0.710148,0.625604


TrainOutput(global_step=70, training_loss=0.5236354282924107, metrics={'train_runtime': 552.947, 'train_samples_per_second': 0.991, 'train_steps_per_second': 0.127, 'total_flos': 2.3100632433106944e+16, 'train_loss': 0.5236354282924107, 'epoch': 2.0})

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,0.518300,0.562938,0.732847,0.659420


TrainOutput(global_step=35, training_loss=0.5182537623814174, metrics={'train_runtime': 276.3167, 'train_samples_per_second': 0.992, 'train_steps_per_second': 0.127, 'total_flos': 1.1527477479419904e+16, 'train_loss': 0.5182537623814174, 'epoch': 1.0})

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
trainer.evaluate()

In [ ]:
# ================================
# Evaluate on Subtask1 + GT labels
# ================================
import re, ast, numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score, jaccard_score

# --- Paths
TEST_Q_PATH = "/content/subtask1_input_test.tsv"  # questions, 1 column, no header
GT_URL_RAW  = "https://raw.githubusercontent.com/hasanhuz/MentalQA/main/ArahealthQA-Track1-MentalQA/TestData/subtask2_output_test.tsv"
OUT_PATH    = "/content/subtask1_with_preds.tsv"

# --- Labels map (for convenience)
LABEL_DESC = {"1": "Information", "2": "Direct Guidance", "3": "Emotional Support"}

# -------------------------
# 1) Load questions (no header)
# -------------------------
df_test = pd.read_csv(TEST_Q_PATH, sep="\t", header=None, names=["question"])

# -------------------------
# 2) Build the same prompts you trained on
# -------------------------
def make_infer_dataset(df, shot_df):
    texts = []
    for _, row in df.iterrows():
        texts.append(build_prompt(row["question"], shot_df))  # <-- your few-shot prompt
    return Dataset.from_dict({"text": texts})

test_ds = make_infer_dataset(df_test, fewshot_sample)

test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

# -------------------------
# 3) Predict with Trainer (logits -> sigmoid -> multi-label)
# -------------------------
pred_out = trainer.predict(test_ds)
logits   = pred_out.predictions

probs    = expit(logits)
pred_bin = (probs >= 0.5).astype(int)   # shape: (N, 3)

# Convert to id lists like ['1','3'] and readable names
# pred_ids = []
# pred_names = []
# for row in pred_bin:
#     labs = [str(i+1) for i, v in enumerate(row) if v == 1]
#     pred_ids.append(", ".join(labs))  # e.g. "1, 3"
#     pred_names.append(", ".join(LABEL_DESC[i] for i in labs))


# 4) load ground truth (each line like: "2, 3" or "1")
gt_raw = pd.read_csv(GT_URL_RAW, header=None, names=["raw"], sep="\t", engine="python")

def parse_gt(line: str):
    # returns list of '1'/'2'/'3'
    return re.findall(r"[123]", str(line))

gt_ids = gt_raw["raw"].apply(parse_gt).tolist()
assert len(gt_ids) == len(df_test), f"GT {len(gt_ids)} != questions {len(df_test)}"

# GT → binary matrix (N x 3)
gt_bin = np.zeros((len(gt_ids), 3), dtype=int)
for i, labs in enumerate(gt_ids):
    for lab in labs:
        gt_bin[i, int(lab)-1] = 1

# 5) metrics (requested: weighted F1 + Jaccard)
f1_weighted   = f1_score(gt_bin, pred_bin, average="weighted", zero_division=0)
jacc_samples  = jaccard_score(gt_bin, pred_bin, average="samples", zero_division=0)
print({"f1_weighted": f1_weighted, "jaccard_samples": jacc_samples})

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

{'f1_weighted': 0.726208279392289, 'jaccard_samples': np.float64(0.6277777777777778)}


In [ ]:
pred_bin

array([[1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1,

In [ ]:
import re, numpy as np, pandas as pd
from datasets import Dataset

# ----------------------------
# Inputs / outputs
# ----------------------------
GITHUB_PAGE_URL = "https://github.com/hasanhuz/MentalQA/blob/main/ArahealthQA-Track1-MentalQA/TestData/Subtask3_input_test.tsv"
OUTPUT_TSV      = "/content/subtask3_with_preds.tsv"
THRESHOLD       = 0.5

# Convert GitHub "blob" URL → raw URL (so pandas can read it)
def to_raw_github(url: str) -> str:
    return re.sub(r"https://github\.com/([^/]+)/([^/]+)/blob/(.+)",
                  r"https://raw.githubusercontent.com/\1/\2/\3", url)

INPUT_URL = to_raw_github(GITHUB_PAGE_URL)

# ----------------------------
# 1) Load questions (no header)
# ----------------------------
df_test = pd.read_csv(INPUT_URL, sep="\t", header=None, names=["question"])
df_test = df_test.dropna(subset=["question"]).reset_index(drop=True)

# ----------------------------
# 2) Build few-shot prompts (same as training)
# ----------------------------
def make_infer_dataset(df, shots_df):
    texts = [build_prompt(q, shots_df) for q in df["question"].tolist()]
    return Dataset.from_dict({"text": texts})

test_ds = make_infer_dataset(df_test, fewshot_sample)


test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

# ----------------------------
# 3) Predict → sigmoid → binary
# ----------------------------
pred_out = trainer.predict(test_ds)                 # logits shape: (N, 3)
logits   = pred_out.predictions
probs    = 1 / (1 + np.exp(-logits))               # sigmoid
pred_bin = (probs >= THRESHOLD).astype(int)        # N x 3

# ----------------------------
# 4) Convert each row to "['1', '2']" format
# ----------------------------
def binrow_to_strlist(row):
    ids = [str(i+1) for i, v in enumerate(row) if v == 1]
    return "[" + ", ".join(f"'{x}'" for x in ids) + "]"

pred_str_lists = [binrow_to_strlist(r) for r in pred_bin]

# ----------------------------
# 5) Save: question + predictions
# ----------------------------
df_out = pd.DataFrame({
    "question": df_test["question"],
    "pred_AS": pred_str_lists
})

df_out.to_csv(OUTPUT_TSV, sep="\t", index=False)
print(f"✅ Saved {len(df_out)} rows → {OUTPUT_TSV}")
print(df_out.head(5))



Map:   0%|          | 0/150 [00:00<?, ? examples/s]

✅ Saved 150 rows → /content/subtask3_with_preds.tsv
                                            question     pred_AS
0  لدي بنت عندها تاخر في النمو العقلي وتاخد دوا و...  ['1', '2']
1  لدي اكتئاب مفاجئ بعد أن تعرضت لصوت رعد مفاجئ و...  ['1', '2']
2  أنا معلمة عربية ولكن بسبب هشاشة النفسية أجد صع...  ['1', '2']
3  عند مقابلة او التحدث إلى مدراء العمل او اشخاص ...  ['1', '2']
4  استعمل الآن prozacوprozalamواجدنفسي ليس لدي شه...  ['1', '2']


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,0.530800,0.585372,0.721083,0.630435
2,0.529600,0.583704,0.707717,0.618357


TrainOutput(global_step=70, training_loss=0.530204336983817, metrics={'train_runtime': 553.3697, 'train_samples_per_second': 0.99, 'train_steps_per_second': 0.126, 'total_flos': 2.3100632433106944e+16, 'train_loss': 0.530204336983817, 'epoch': 2.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.5837036371231079,
 'eval_weighted_f1': 0.7077169050238358,
 'eval_jaccard': 0.6183574879227053,
 'eval_runtime': 55.1696,
 'eval_samples_per_second': 1.251,
 'eval_steps_per_second': 0.163,
 'epoch': 2.0}

In [ ]:
# ================================
# Evaluate on Subtask1 + GT labels
# ================================
import re, ast, numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score, jaccard_score

# --- Paths
TEST_Q_PATH = "/content/subtask1_input_test.tsv"  # questions, 1 column, no header
GT_URL_RAW  = "https://raw.githubusercontent.com/hasanhuz/MentalQA/main/ArahealthQA-Track1-MentalQA/TestData/subtask2_output_test.tsv"
OUT_PATH    = "/content/subtask1_with_preds.tsv"

# --- Labels map (for convenience)
LABEL_DESC = {"1": "Information", "2": "Direct Guidance", "3": "Emotional Support"}

# -------------------------
# 1) Load questions (no header)
# -------------------------
df_test = pd.read_csv(TEST_Q_PATH, sep="\t", header=None, names=["question"])

# -------------------------
# 2) Build the same prompts you trained on
# -------------------------
def make_infer_dataset(df, shot_df):
    texts = []
    for _, row in df.iterrows():
        texts.append(build_prompt(row["question"], shot_df))  # <-- your few-shot prompt
    return Dataset.from_dict({"text": texts})

test_ds = make_infer_dataset(df_test, fewshot_sample)

test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

# -------------------------
# 3) Predict with Trainer (logits -> sigmoid -> multi-label)
# -------------------------
pred_out = trainer.predict(test_ds)
logits   = pred_out.predictions

probs    = expit(logits)
pred_bin = (probs >= 0.5).astype(int)   # shape: (N, 3)

# Convert to id lists like ['1','3'] and readable names
# pred_ids = []
# pred_names = []
# for row in pred_bin:
#     labs = [str(i+1) for i, v in enumerate(row) if v == 1]
#     pred_ids.append(", ".join(labs))  # e.g. "1, 3"
#     pred_names.append(", ".join(LABEL_DESC[i] for i in labs))


# 4) load ground truth (each line like: "2, 3" or "1")
gt_raw = pd.read_csv(GT_URL_RAW, header=None, names=["raw"], sep="\t", engine="python")

def parse_gt(line: str):
    # returns list of '1'/'2'/'3'
    return re.findall(r"[123]", str(line))

gt_ids = gt_raw["raw"].apply(parse_gt).tolist()
assert len(gt_ids) == len(df_test), f"GT {len(gt_ids)} != questions {len(df_test)}"

# GT → binary matrix (N x 3)
gt_bin = np.zeros((len(gt_ids), 3), dtype=int)
for i, labs in enumerate(gt_ids):
    for lab in labs:
        gt_bin[i, int(lab)-1] = 1

# 5) metrics (requested: weighted F1 + Jaccard)
f1_weighted   = f1_score(gt_bin, pred_bin, average="weighted", zero_division=0)
jacc_samples  = jaccard_score(gt_bin, pred_bin, average="samples", zero_division=0)
print({"f1_weighted": f1_weighted, "jaccard_samples": jacc_samples})

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

{'f1_weighted': 0.7135498421982481, 'jaccard_samples': np.float64(0.6122222222222223)}


In [ ]:
pred_bin

array([[1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1,

In [ ]:
import re, numpy as np, pandas as pd
from datasets import Dataset

# ----------------------------
# Inputs / outputs
# ----------------------------
GITHUB_PAGE_URL = "https://github.com/hasanhuz/MentalQA/blob/main/ArahealthQA-Track1-MentalQA/TestData/Subtask3_input_test.tsv"
OUTPUT_TSV      = "/content/subtask3_with_preds.tsv"
THRESHOLD       = 0.5

# Convert GitHub "blob" URL → raw URL (so pandas can read it)
def to_raw_github(url: str) -> str:
    return re.sub(r"https://github\.com/([^/]+)/([^/]+)/blob/(.+)",
                  r"https://raw.githubusercontent.com/\1/\2/\3", url)

INPUT_URL = to_raw_github(GITHUB_PAGE_URL)

# ----------------------------
# 1) Load questions (no header)
# ----------------------------
df_test = pd.read_csv(INPUT_URL, sep="\t", header=None, names=["question"])
df_test = df_test.dropna(subset=["question"]).reset_index(drop=True)

# ----------------------------
# 2) Build few-shot prompts (same as training)
# ----------------------------
def make_infer_dataset(df, shots_df):
    texts = [build_prompt(q, shots_df) for q in df["question"].tolist()]
    return Dataset.from_dict({"text": texts})

test_ds = make_infer_dataset(df_test, fewshot_sample)


test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

# ----------------------------
# 3) Predict → sigmoid → binary
# ----------------------------
pred_out = trainer.predict(test_ds)                 # logits shape: (N, 3)
logits   = pred_out.predictions
probs    = 1 / (1 + np.exp(-logits))               # sigmoid
pred_bin = (probs >= THRESHOLD).astype(int)        # N x 3

# ----------------------------
# 4) Convert each row to "['1', '2']" format
# ----------------------------
def binrow_to_strlist(row):
    ids = [str(i+1) for i, v in enumerate(row) if v == 1]
    return "[" + ", ".join(f"'{x}'" for x in ids) + "]"

pred_str_lists = [binrow_to_strlist(r) for r in pred_bin]

# ----------------------------
# 5) Save: question + predictions
# ----------------------------
df_out = pd.DataFrame({
    "question": df_test["question"],
    "pred_AS": pred_str_lists
})

df_out.to_csv(OUTPUT_TSV, sep="\t", index=False)
print(f"✅ Saved {len(df_out)} rows → {OUTPUT_TSV}")
print(df_out.head(5))



Map:   0%|          | 0/150 [00:00<?, ? examples/s]

✅ Saved 150 rows → /content/subtask3_with_preds.tsv
                                            question     pred_AS
0  لدي بنت عندها تاخر في النمو العقلي وتاخد دوا و...  ['1', '2']
1  لدي اكتئاب مفاجئ بعد أن تعرضت لصوت رعد مفاجئ و...  ['1', '2']
2  أنا معلمة عربية ولكن بسبب هشاشة النفسية أجد صع...  ['1', '2']
3  عند مقابلة او التحدث إلى مدراء العمل او اشخاص ...  ['1', '2']
4  استعمل الآن prozacوprozalamواجدنفسي ليس لدي شه...  ['1', '2']


In [ ]:
trainer.evaluate()

{'eval_loss': 0.5828502178192139,
 'eval_weighted_f1': 0.7100079108234448,
 'eval_jaccard': 0.6183574879227053,
 'eval_runtime': 55.2053,
 'eval_samples_per_second': 1.25,
 'eval_steps_per_second': 0.163,
 'epoch': 2.0}

# 2 Epochs

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,1.064200,0.783045,0.763561,0.521739
2,0.750000,0.680552,0.760026,0.543478


TrainOutput(global_step=70, training_loss=0.9070773260934012, metrics={'train_runtime': 554.1228, 'train_samples_per_second': 0.989, 'train_steps_per_second': 0.126, 'total_flos': 2.3100632433106944e+16, 'train_loss': 0.9070773260934012, 'epoch': 2.0})

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,0.600100,0.576301,0.721083,0.630435
2,0.548300,0.572845,0.714593,0.625604


TrainOutput(global_step=70, training_loss=0.5742235456194197, metrics={'train_runtime': 552.6516, 'train_samples_per_second': 0.992, 'train_steps_per_second': 0.127, 'total_flos': 2.3100632433106944e+16, 'train_loss': 0.5742235456194197, 'epoch': 2.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.5728449821472168,
 'eval_weighted_f1': 0.7145925925925926,
 'eval_jaccard': 0.6256038647342995,
 'eval_runtime': 55.1278,
 'eval_samples_per_second': 1.252,
 'eval_steps_per_second': 0.163,
 'epoch': 2.0}

In [ ]:
# 6) save predictions in SAME format as GT (one line per sample like: "2, 3" or "1")
def binrow_to_ids(row):
    return [str(i+1) for i, v in enumerate(row) if v == 1]

pred_lines = [", ".join(binrow_to_ids(row)) for row in pred_bin]  # empty string if none
pd.Series(pred_lines).to_csv(PRED_OUT, index=False, header=False)
print(f"✅ Wrote predictions to {PRED_OUT} (format matches ground truth)")

# Final

In [ ]:
# ================================
# Evaluate on Subtask1 + GT labels
# ================================
import re, ast, numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score, jaccard_score

# --- Paths
TEST_Q_PATH = "/content/subtask1_input_test.tsv"  # questions, 1 column, no header
GT_URL_RAW  = "https://raw.githubusercontent.com/hasanhuz/MentalQA/main/ArahealthQA-Track1-MentalQA/TestData/subtask2_output_test.tsv"
OUT_PATH    = "/content/subtask1_with_preds.tsv"

# --- Labels map (for convenience)
LABEL_DESC = {"1": "Information", "2": "Direct Guidance", "3": "Emotional Support"}

# -------------------------
# 1) Load questions (no header)
# -------------------------
df_test = pd.read_csv(TEST_Q_PATH, sep="\t", header=None, names=["question"])

# -------------------------
# 2) Build the same prompts you trained on
# -------------------------
def make_infer_dataset(df, shot_df):
    texts = []
    for _, row in df.iterrows():
        texts.append(build_prompt(row["question"], shot_df))  # <-- your few-shot prompt
    return Dataset.from_dict({"text": texts})

test_ds = make_infer_dataset(df_test, fewshot_sample)

test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

# -------------------------
# 3) Predict with Trainer (logits -> sigmoid -> multi-label)
# -------------------------
pred_out = trainer.predict(test_ds)
logits   = pred_out.predictions

probs    = expit(logits)
pred_bin = (probs >= 0.5).astype(int)   # shape: (N, 3)

# Convert to id lists like ['1','3'] and readable names
# pred_ids = []
# pred_names = []
# for row in pred_bin:
#     labs = [str(i+1) for i, v in enumerate(row) if v == 1]
#     pred_ids.append(", ".join(labs))  # e.g. "1, 3"
#     pred_names.append(", ".join(LABEL_DESC[i] for i in labs))


# 4) load ground truth (each line like: "2, 3" or "1")
gt_raw = pd.read_csv(GT_URL_RAW, header=None, names=["raw"], sep="\t", engine="python")

def parse_gt(line: str):
    # returns list of '1'/'2'/'3'
    return re.findall(r"[123]", str(line))

gt_ids = gt_raw["raw"].apply(parse_gt).tolist()
assert len(gt_ids) == len(df_test), f"GT {len(gt_ids)} != questions {len(df_test)}"

# GT → binary matrix (N x 3)
gt_bin = np.zeros((len(gt_ids), 3), dtype=int)
for i, labs in enumerate(gt_ids):
    for lab in labs:
        gt_bin[i, int(lab)-1] = 1

# 5) metrics (requested: weighted F1 + Jaccard)
f1_weighted   = f1_score(gt_bin, pred_bin, average="weighted", zero_division=0)
jacc_samples  = jaccard_score(gt_bin, pred_bin, average="samples", zero_division=0)
print({"f1_weighted": f1_weighted, "jaccard_samples": jacc_samples})

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

{'f1_weighted': 0.7195064020763766, 'jaccard_samples': np.float64(0.6211111111111111)}


In [ ]:
pred_bin

array([[1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1,

In [ ]:
trainer.evaluate()

{'eval_loss': 0.6805523037910461,
 'eval_weighted_f1': 0.7600260827072262,
 'eval_jaccard': 0.5434782608695652,
 'eval_runtime': 55.1331,
 'eval_samples_per_second': 1.252,
 'eval_steps_per_second': 0.163,
 'epoch': 2.0}

In [ ]:
# ================================
# Evaluate on Subtask1 + GT labels
# ================================
import re, ast, numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score, jaccard_score

# --- Paths
TEST_Q_PATH = "/content/subtask1_input_test.tsv"  # questions, 1 column, no header
GT_URL_RAW  = "https://raw.githubusercontent.com/hasanhuz/MentalQA/main/ArahealthQA-Track1-MentalQA/TestData/subtask2_output_test.tsv"
OUT_PATH    = "/content/subtask1_with_preds.tsv"

# --- Labels map (for convenience)
LABEL_DESC = {"1": "Information", "2": "Direct Guidance", "3": "Emotional Support"}

# -------------------------
# 1) Load questions (no header)
# -------------------------
df_test = pd.read_csv(TEST_Q_PATH, sep="\t", header=None, names=["question"])

# -------------------------
# 2) Build the same prompts you trained on
# -------------------------
def make_infer_dataset(df, shot_df):
    texts = []
    for _, row in df.iterrows():
        texts.append(build_prompt(row["question"], shot_df))  # <-- your few-shot prompt
    return Dataset.from_dict({"text": texts})

test_ds = make_infer_dataset(df_test, fewshot_sample)

test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

# -------------------------
# 3) Predict with Trainer (logits -> sigmoid -> multi-label)
# -------------------------
pred_out = trainer.predict(test_ds)
logits   = pred_out.predictions

probs    = expit(logits)
pred_bin = (probs >= 0.5).astype(int)   # shape: (N, 3)

# Convert to id lists like ['1','3'] and readable names
# pred_ids = []
# pred_names = []
# for row in pred_bin:
#     labs = [str(i+1) for i, v in enumerate(row) if v == 1]
#     pred_ids.append(", ".join(labs))  # e.g. "1, 3"
#     pred_names.append(", ".join(LABEL_DESC[i] for i in labs))


# 4) load ground truth (each line like: "2, 3" or "1")
gt_raw = pd.read_csv(GT_URL_RAW, header=None, names=["raw"], sep="\t", engine="python")

def parse_gt(line: str):
    # returns list of '1'/'2'/'3'
    return re.findall(r"[123]", str(line))

gt_ids = gt_raw["raw"].apply(parse_gt).tolist()
assert len(gt_ids) == len(df_test), f"GT {len(gt_ids)} != questions {len(df_test)}"

# GT → binary matrix (N x 3)
gt_bin = np.zeros((len(gt_ids), 3), dtype=int)
for i, labs in enumerate(gt_ids):
    for lab in labs:
        gt_bin[i, int(lab)-1] = 1

# 5) metrics (requested: weighted F1 + Jaccard)
f1_weighted   = f1_score(gt_bin, pred_bin, average="weighted", zero_division=0)
jacc_samples  = jaccard_score(gt_bin, pred_bin, average="samples", zero_division=0)
print({"f1_weighted": f1_weighted, "jaccard_samples": jacc_samples})

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

{'f1_weighted': 0.7436562486899914, 'jaccard_samples': np.float64(0.49777777777777793)}


In [ ]:
pred_bin

array([[1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1,

In [ ]:
import re, numpy as np, pandas as pd
from datasets import Dataset

# ----------------------------
# Inputs / outputs
# ----------------------------
GITHUB_PAGE_URL = "https://github.com/hasanhuz/MentalQA/blob/main/ArahealthQA-Track1-MentalQA/TestData/Subtask3_input_test.tsv"
OUTPUT_TSV      = "/content/subtask3_with_preds.tsv"
THRESHOLD       = 0.5

# Convert GitHub "blob" URL → raw URL (so pandas can read it)
def to_raw_github(url: str) -> str:
    return re.sub(r"https://github\.com/([^/]+)/([^/]+)/blob/(.+)",
                  r"https://raw.githubusercontent.com/\1/\2/\3", url)

INPUT_URL = to_raw_github(GITHUB_PAGE_URL)

# ----------------------------
# 1) Load questions (no header)
# ----------------------------
df_test = pd.read_csv(INPUT_URL, sep="\t", header=None, names=["question"])
df_test = df_test.dropna(subset=["question"]).reset_index(drop=True)

# ----------------------------
# 2) Build few-shot prompts (same as training)
# ----------------------------
def make_infer_dataset(df, shots_df):
    texts = [build_prompt(q, shots_df) for q in df["question"].tolist()]
    return Dataset.from_dict({"text": texts})

test_ds = make_infer_dataset(df_test, fewshot_sample)



In [ ]:
print(test_ds['text'][0])

You are an assistant that determines the appropriate response style(s) for a given user query in a mental health context. A query may require one or more of the following styles:
- Information: Provide factual information, resources, or requests for information.
- Direct Guidance: Offer suggestions, instructions, or advice, including actions the user should take to address their situation.
- Emotional Support: Provide approval, reassurance, empathy, or other forms of non-informational emotional support.

Classify the input into all applicable styles based on its content.

### Example
User's Query: "ودي استفسر بخصوص بروزاك 20 كتبه لي اخصائي نفسي كنت استخدم بروكسات 20 و10 ووقفته انا من نفسي لانه سبب كسل ونوم ووزن زايذ راجعت دكتور نفسي اعطاني بروزاك 20 لكن لم استخدمه"
Needed Answer Style(s): Information, Direct Guidance

### Example
User's Query: "ماهو افضل علاج دوائي للاكتئاب وماهي اعراضه الجانبيه علما اني لا استطيع الذهاب لطبيب نفسي للعلاج"
Needed Answer Style(s): Direct Guidance

### E

In [ ]:
torch.cuda.empty_cache()

In [ ]:

test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

# ----------------------------
# 3) Predict → sigmoid → binary
# ----------------------------
pred_out = trainer.predict(test_ds)                 # logits shape: (N, 3)
logits   = pred_out.predictions
probs    = 1 / (1 + np.exp(-logits))               # sigmoid
pred_bin = (probs >= THRESHOLD).astype(int)        # N x 3

# ----------------------------
# 4) Convert each row to "['1', '2']" format
# ----------------------------
def binrow_to_strlist(row):
    ids = [str(i+1) for i, v in enumerate(row) if v == 1]
    return "[" + ", ".join(f"'{x}'" for x in ids) + "]"

pred_str_lists = [binrow_to_strlist(r) for r in pred_bin]

# ----------------------------
# 5) Save: question + predictions
# ----------------------------
df_out = pd.DataFrame({
    "question": df_test["question"],
    "pred_AS": pred_str_lists
})

df_out.to_csv(OUTPUT_TSV, sep="\t", index=False)
print(f"✅ Saved {len(df_out)} rows → {OUTPUT_TSV}")
print(df_out.head(5))


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

✅ Saved 150 rows → /content/subtask3_with_preds.tsv
                                            question     pred_AS
0  لدي بنت عندها تاخر في النمو العقلي وتاخد دوا و...  ['1', '2']
1  لدي اكتئاب مفاجئ بعد أن تعرضت لصوت رعد مفاجئ و...  ['1', '2']
2  أنا معلمة عربية ولكن بسبب هشاشة النفسية أجد صع...  ['1', '2']
3  عند مقابلة او التحدث إلى مدراء العمل او اشخاص ...  ['1', '2']
4  استعمل الآن prozacوprozalamواجدنفسي ليس لدي شه...  ['1', '2']


In [ ]:
pred_bin

array([[1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1,

# 1 Epoch

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,1.120000,0.921518,0.763561,0.521739


TrainOutput(global_step=35, training_loss=1.1200305393763952, metrics={'train_runtime': 277.36, 'train_samples_per_second': 0.988, 'train_steps_per_second': 0.126, 'total_flos': 1.1527477479419904e+16, 'train_loss': 1.1200305393763952, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.9215183854103088,
 'eval_weighted_f1': 0.7635612767238784,
 'eval_jaccard': 0.5217391304347826,
 'eval_runtime': 55.1356,
 'eval_samples_per_second': 1.251,
 'eval_steps_per_second': 0.163,
 'epoch': 1.0}

In [ ]:
# ================================
# Evaluate on Subtask1 + GT labels
# ================================
import re, ast, numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score, jaccard_score

# --- Paths
TEST_Q_PATH = "/content/subtask1_input_test.tsv"  # questions, 1 column, no header
GT_URL_RAW  = "https://raw.githubusercontent.com/hasanhuz/MentalQA/main/ArahealthQA-Track1-MentalQA/TestData/subtask2_output_test.tsv"
OUT_PATH    = "/content/subtask1_with_preds.tsv"

# --- Labels map (for convenience)
LABEL_DESC = {"1": "Information", "2": "Direct Guidance", "3": "Emotional Support"}

# -------------------------
# 1) Load questions (no header)
# -------------------------
df_test = pd.read_csv(TEST_Q_PATH, sep="\t", header=None, names=["question"])

# -------------------------
# 2) Build the same prompts you trained on
# -------------------------
def make_infer_dataset(df, shot_df):
    texts = []
    for _, row in df.iterrows():
        texts.append(build_prompt(row["question"], shot_df))  # <-- your few-shot prompt
    return Dataset.from_dict({"text": texts})

test_ds = make_infer_dataset(df_test, fewshot_sample)



In [ ]:
test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"])


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [ ]:
test_ds

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 150
})

In [ ]:
# -------------------------
# 3) Predict with Trainer (logits -> sigmoid -> multi-label)
# -------------------------
pred_out = trainer.predict(test_ds)

In [ ]:
logits   = pred_out.predictions

In [ ]:
logits[0]

array([1.033213 , 1.1811624, 1.4995406], dtype=float32)

In [ ]:
probs    = 1 / (1 + np.exp(-logits))
pred_bin = (probs >= 0.5).astype(int)   # shape: (N, 3)

In [ ]:
pred_bin

array([[1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1,

In [ ]:
probs

array([[0.7375384 , 0.76515675, 0.8175059 ],
       [0.6408894 , 0.7914637 , 0.8469462 ],
       [0.7128441 , 0.80016214, 0.83785695],
       [0.6825078 , 0.74827   , 0.84787107],
       [0.6517742 , 0.7892785 , 0.7836324 ],
       [0.5679727 , 0.71289957, 0.86764795],
       [0.53829354, 0.8230707 , 0.9459794 ],
       [0.5931598 , 0.72890806, 0.7936546 ],
       [0.67912954, 0.82424366, 0.8882006 ],
       [0.6274232 , 0.78691983, 0.86799306],
       [0.5940059 , 0.7702512 , 0.7915632 ],
       [0.62556744, 0.7670531 , 0.76154757],
       [0.7770669 , 0.81239766, 0.7442722 ],
       [0.59916735, 0.766223  , 0.8363643 ],
       [0.60576034, 0.78826034, 0.8972761 ],
       [0.5912925 , 0.7738126 , 0.6556905 ],
       [0.7400873 , 0.67868817, 0.6656801 ],
       [0.6136177 , 0.7721556 , 0.788724  ],
       [0.6828585 , 0.8436463 , 0.9122777 ],
       [0.67108625, 0.7404722 , 0.78149194],
       [0.6383288 , 0.69717455, 0.8222284 ],
       [0.61468315, 0.7949967 , 0.86476284],
       [0.

In [ ]:
pred_bin

array([[1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1,

In [ ]:
# Convert to id lists like ['1','3'] and readable names
pred_ids = []
pred_names = []
for row in pred_bin:
    labs = [str(i+1) for i, v in enumerate(row) if v == 1]
    pred_ids.append(", ".join(labs))  # e.g. "1, 3"
    pred_names.append(", ".join(LABEL_DESC[i] for i in labs))

In [ ]:
gt_raw

,raw
0,"2, 3"
1,"1, 2"
2,1
3,"1, 2, 3"
4,"1, 2"
...,...
145,1
146,1
147,2
148,1


In [ ]:
gt_ids

[['2', '3'],
 ['1', '2'],
 ['1'],
 ['1', '2', '3'],
 ['1', '2'],
 ['1', '2'],
 ['1'],
 ['1', '2'],
 ['1', '2'],
 ['1', '3'],
 ['1', '2', '3'],
 ['1', '2', '3'],
 ['1'],
 ['1'],
 ['1'],
 ['2', '3'],
 ['2', '3'],
 ['1', '3'],
 ['1', '2'],
 ['1', '2'],
 ['1', '2'],
 ['1', '2'],
 ['2'],
 ['1', '2'],
 ['1', '2'],
 ['1', '2'],
 ['1'],
 ['1', '2'],
 ['1'],
 ['1', '3'],
 ['1', '2'],
 ['2'],
 ['1', '2'],
 ['1', '2', '3'],
 ['1'],
 ['1', '2'],
 ['1'],
 ['1', '2'],
 ['1', '2'],
 ['1', '3'],
 ['1', '3'],
 ['1', '2', '3'],
 ['1'],
 ['1'],
 ['2', '3'],
 ['1', '2', '3'],
 ['1', '2'],
 ['1', '2'],
 ['1'],
 ['1', '2'],
 ['1'],
 ['1'],
 ['1', '2', '3'],
 ['2'],
 ['1', '2'],
 ['1'],
 ['1'],
 ['1'],
 ['1', '3'],
 ['1', '2'],
 ['1'],
 ['1', '2'],
 ['2'],
 ['1'],
 ['1', '2'],
 ['2'],
 ['1', '2'],
 ['1', '2'],
 ['1', '2'],
 ['2'],
 ['1', '2'],
 ['1'],
 ['1', '2'],
 ['2'],
 ['1', '2'],
 ['2'],
 ['1'],
 ['1', '2'],
 ['1'],
 ['1'],
 ['1'],
 ['2'],
 ['2'],
 ['1', '2'],
 ['1'],
 ['1', '2'],
 ['1', '2'],
 ['1', '2

In [ ]:
gt_bin

array([[0, 1, 1],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 1],
       [1, 1, 1],
       [1, 1, 1],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 1],
       [0, 1, 1],
       [1, 0, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [0, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 1],
       [1, 1, 0],
       [0, 1, 0],
       [1, 1, 0],
       [1, 1, 1],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 1],
       [1, 0, 1],
       [1, 1, 1],
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 1],
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 1, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 1, 1],
       [0, 1, 0],
       [1, 1, 0],
       [1,

In [ ]:
# 4) load ground truth (each line like: "2, 3" or "1")
gt_raw = pd.read_csv(GT_URL_RAW, header=None, names=["raw"], sep="\t", engine="python")

def parse_gt(line: str):
    # returns list of '1'/'2'/'3'
    return re.findall(r"[123]", str(line))

gt_ids = gt_raw["raw"].apply(parse_gt).tolist()
assert len(gt_ids) == len(df_test), f"GT {len(gt_ids)} != questions {len(df_test)}"

# GT → binary matrix (N x 3)
gt_bin = np.zeros((len(gt_ids), 3), dtype=int)
for i, labs in enumerate(gt_ids):
    for lab in labs:
        gt_bin[i, int(lab)-1] = 1

# 5) metrics (requested: weighted F1 + Jaccard)
f1_weighted   = f1_score(gt_bin, pred_bin, average="weighted", zero_division=0)
jacc_samples  = jaccard_score(gt_bin, pred_bin, average="samples", zero_division=0)
print({"f1_weighted": f1_weighted, "jaccard_samples": jacc_samples})

{'f1_weighted': 0.7513464543966549, 'jaccard_samples': np.float64(0.48)}


In [ ]:
# -------------------------
# 4) Load Ground Truth from GitHub (robust parser)
# -------------------------
# The file contains one combination per line like:
# 2, 3
# 1
# 1, 2, 3
gt_raw = pd.read_csv(GT_URL_RAW, header=None, names=["raw"], sep="\n", engine="python")
def parse_gt(line: str):
    return [m for m in re.findall(r"[123]", str(line))]

gt_ids = gt_raw["raw"].apply(parse_gt).tolist()

# Sanity check lengths
assert len(gt_ids) == len(df_test), f"GT size {len(gt_ids)} != questions {len(df_test)}"

# Convert GT to binary matrix
gt_bin = np.zeros((len(gt_ids), 3), dtype=int)
for i, labs in enumerate(gt_ids):
    for lab in labs:
        gt_bin[i, int(lab)-1] = 1

# -------------------------
# 5) Metrics
# -------------------------
weighted_f1 = f1_score(gt_bin, pred_bin, average="weighted", zero_division=0)
jacc_samples = jaccard_score(gt_bin, pred_bin, average="samples", zero_division=0)

print({
    "f1_weighted": weighted_f1,
    "jaccard_samples": jacc_samples,
})

ValueError: Specified \n as separator or delimiter. This forces the python engine which does not accept a line terminator. Hence it is not allowed to use the line terminator as separator.

In [ ]:



# -------------------------
# 3) Predict with Trainer (logits -> sigmoid -> multi-label)
# -------------------------
pred_out = trainer.predict(test_ds)   # .predictions is (N, 3) logits
logits   = pred_out.predictions
probs    = 1 / (1 + np.exp(-logits))
pred_bin = (probs >= 0.5).astype(int)   # shape: (N, 3)

# Convert to id lists like ['1','3'] and readable names
pred_ids = []
pred_names = []
for row in pred_bin:
    labs = [str(i+1) for i, v in enumerate(row) if v == 1]
    pred_ids.append(", ".join(labs))  # e.g. "1, 3"
    pred_names.append(", ".join(LABEL_DESC[i] for i in labs))

# -------------------------
# 4) Load Ground Truth from GitHub (robust parser)
# -------------------------
# The file contains one combination per line like:
# 2, 3
# 1
# 1, 2, 3
gt_raw = pd.read_csv(GT_URL_RAW, header=None, names=["raw"], sep="\n", engine="python")
def parse_gt(line: str):
    return [m for m in re.findall(r"[123]", str(line))]

gt_ids = gt_raw["raw"].apply(parse_gt).tolist()

# Sanity check lengths
assert len(gt_ids) == len(df_test), f"GT size {len(gt_ids)} != questions {len(df_test)}"

# Convert GT to binary matrix
gt_bin = np.zeros((len(gt_ids), 3), dtype=int)
for i, labs in enumerate(gt_ids):
    for lab in labs:
        gt_bin[i, int(lab)-1] = 1

# -------------------------
# 5) Metrics
# -------------------------
weighted_f1 = f1_score(gt_bin, pred_bin, average="weighted", zero_division=0)
jacc_samples = jaccard_score(gt_bin, pred_bin, average="samples", zero_division=0)

print({
    "f1_weighted": weighted_f1,
    "jaccard_samples": jacc_samples,
})

# -------------------------
# 6) Save predictions alongside questions
# -------------------------
df_test["pred_AS"]       = pred_ids          # e.g. "1, 3"
df_test["pred_AS_names"] = pred_names        # e.g. "Information, Emotional Support"
df_test.to_csv(OUT_PATH, sep="\t", index=False)
print(f"✅ Saved predictions to {OUT_PATH}")


In [ ]:
import pandas as pd, torch, tqdm, re

TEST_IN  = "/content/subtask1_input_test.tsv"        # no header
TEST_OUT = "/content/subtask1_with_preds.tsv"

# -------------------------
# 1  give it a column name
# -------------------------
df_test = pd.read_csv(
    TEST_IN,
    sep="\t",
    header=None,            # ← tells pandas “there is no header row”
    names=["question"]      # ← create a single column called   question
)

# -------------------------
# 2  run generation → parse
# -------------------------
LABEL_DESC = {"1": "Information",
              "2": "Direct Guidance",
              "3": "Emotional Support"}
REGEX = {k: re.compile(re.escape(v), re.IGNORECASE) for k, v in LABEL_DESC.items()}

def parse_styles(text):
    return {lab for lab, rgx in REGEX.items() if rgx.search(text)}

pred_ids, pred_names = [], []
model.eval()

for _, row in tqdm.tqdm(df_test.iterrows(), total=len(df_test)):
    prompt = build_prompt(row["question"], fewshot_sample)
    in_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        gen_ids = model.generate(**in_ids, max_new_tokens=15, do_sample=False)

    gen_txt = tokenizer.decode(
        gen_ids[0][in_ids["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    labs = parse_styles(gen_txt)
    pred_ids.append(str(sorted(labs)))                          # e.g. "['1','3']"
    pred_names.append(", ".join(LABEL_DESC[l] for l in labs))   # readable

df_test["pred_AS"]       = pred_ids
df_test["pred_AS_names"] = pred_names

df_test.to_csv(TEST_OUT, sep="\t", index=False)
print(f"✅ predictions written to {TEST_OUT}")


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,1.025000,0.701139,0.757359,0.528986
2,0.628300,0.572075,0.721083,0.630435
3,0.543100,0.576235,0.700566,0.603865
4,0.531200,0.581358,0.721083,0.630435
5,0.536900,0.583594,0.710008,0.618357


KeyboardInterrupt: 

In [ ]:
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Weighted F1,Jaccard
1,1.025000,0.701139,0.757359,0.528986
2,0.628300,0.572075,0.721083,0.630435
3,0.543100,0.576235,0.700566,0.603865
4,0.531200,0.581358,0.721083,0.630435
5,0.536900,0.582879,0.710008,0.618357


{'eval_loss': 0.5828785300254822,
 'eval_weighted_f1': 0.7100079108234448,
 'eval_jaccard': 0.6183574879227053}

In [ ]:



# ----------------------------------------------------------
# 6.  TRAINER
# ----------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# ----------------------------------------------------------
# 7.  LAUNCH TRAINING
# ----------------------------------------------------------
trainer.train()


In [ ]:
# ================================================================
# FEW-SHOT   MULTI-LABEL   STYLE-CLASSIFIER   (Qwen-3-8B)
# ================================================================
# !pip install -q scikit-learn datasets



# ----------------------------------------------------------
# 1.  UTILS
# ----------------------------------------------------------
def multihot(label_list):
    """Convert list of str → multi-hot vec."""
    y = [0]*len(LABEL_ID)
    for lab in label_list:
        if lab in LABEL_ID: y[LABEL_ID[lab]] = 1
    return y

def build_prompt(query, shots_df, k=FEWSHOT_K):
    """Few-shot prompt with *k* exemplars."""
    exemplars = shots_df.sample(
        n=min(k, len(shots_df)),
        random_state=random.randint(0, 10**6)
    )

    parts = [
      "You are an assistant that decides which response style(s)"
      " the user needs. Possible styles are: "
      "Information, Direct Guidance, Emotional Support.\n"
    ]

    # ----- exemplars -----
    for _, row in exemplars.iterrows():
        lbls = ast.literal_eval(row["final_AS"]) if isinstance(row["final_AS"], str) else row["final_AS"]
        lbl_names = ", ".join(LABEL_DESC[l] for l in lbls)
        parts.append(
            f"### Example\nUser: \"{row['question']}\"\n"
            f"Needed Style(s): {lbl_names}\n"
        )

    # ----- target -----
    parts.append(
        "### Task\n"
        f"User: \"{query}\"\n"
        "Needed Style(s):"
    )
    return "\n".join(parts)

def make_hf_dataset(df, shot_df):
    """HF Dataset → {text, labels}"""
    texts, ys = [], []
    for _, row in df.iterrows():
        label_raw = ast.literal_eval(row["final_AS"]) if isinstance(row["final_AS"], str) else row["final_AS"]
        texts.append(build_prompt(row["question"], shot_df))
        ys.append(multihot(label_raw))
    return Dataset.from_dict({"text": texts, "labels": ys})

# ----------------------------------------------------------
# 2.  DATA
# ----------------------------------------------------------
# ▸ ASSUMPTION: train_df, val_df, fewshot_sample already exist
train_dataset = make_hf_dataset(train_df, fewshot_sample)
val_dataset   = make_hf_dataset(val_df,   fewshot_sample)

def tok_fn(batch):
    return tokenizer(batch["text"],
                     truncation=True,
                     padding="max_length",
                     max_length=768)

train_dataset = train_dataset.map(tok_fn, batched=True, remove_columns=["text"])
val_dataset   = val_dataset.map(tok_fn,   batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")

# ----------------------------------------------------------
# 3.  MODEL
# ----------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL,
            num_labels=len(LABEL_ID),
            problem_type="multi_label_classification",
            trust_remote_code=True
        ).to(DEVICE)

# ----------------------------------------------------------
# 4.  METRICS
# ----------------------------------------------------------
def compute_metrics(eval_pred):
    logits, gold = eval_pred
    probs = 1 / (1 + np.exp(-logits))      # sigmoid
    preds = (probs >= 0.5).astype(int)

    f1  = f1_score(gold, preds, average="macro", zero_division=0)
    jac = jaccard_score(gold, preds, average="samples", zero_division=0)
    return {"macro_f1": f1, "jaccard": jac}

# ----------------------------------------------------------
# 5.  TRAIN ARGS
# ----------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=1,     # Qwen-8B is heavy → keep small
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=20,
    seed=SEED,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

# ----------------------------------------------------------
# 6.  TRAINER
# ----------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# ----------------------------------------------------------
# 7.  LAUNCH TRAINING
# ----------------------------------------------------------
trainer.train()


In [ ]:
# MODEL_ID = "Qwen/Qwen3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True
# )

# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=15,
#     return_full_text=False
# )




In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Define LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # works for most LLaMA-like models
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach LoRA adapters
model = get_peft_model(model, lora_config)


In [ ]:
QT_LABELS = {
    "1": "Information",
    "2": "Direct Guidance",
    "3": "Emotional Support"
}


In [ ]:
# ===========================================================
# Qwen‑3‑8B  |  Few‑Shot  |  Multi‑Label Style  |  Gen+Parse
# ===========================================================
# !pip install -q scikit-learn datasets bitsandbytes accelerate peft

import os, ast, re, random, torch, numpy as np, pandas as pd
from datasets import Dataset
from sklearn.metrics import f1_score, jaccard_score

from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, TrainingArguments, Trainer)
from transformers import DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ------------------------------------------------------------------
# 0.  ENV & CONSTANTS
# ------------------------------------------------------------------
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

SEED       = 42
FEWSHOT_K  = 3
MODEL_ID   = "Qwen/Qwen3-8B"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LABEL_DESC = {
    "1": "Information",
    "2": "Direct Guidance",
    "3": "Emotional Support"
}
# build regex once for parsing
LABEL_REGEX = {
    lab: re.compile(re.escape(name), re.IGNORECASE)
    for lab, name in LABEL_DESC.items()
}

# ------------------------------------------------------------------
# 1.  MODEL  (4‑bit + LoRA)
# ------------------------------------------------------------------
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token   # Qwen has no pad_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True
)

base_model = prepare_model_for_kbit_training(base_model)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, lora_cfg)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:

# ------------------------------------------------------------------
# 2.  PROMPT / TARGET BUILDERS
# ------------------------------------------------------------------
def multihot(label_list):
    vec = [0, 0, 0]
    for lab in label_list:
        if lab in ("1", "2", "3"):
            vec[int(lab)-1] = 1
    return vec

def build_prompt(query: str, shots_df: pd.DataFrame, k=FEWSHOT_K) -> str:
    """Few‑shot prompt WITHOUT the answer to the target query."""
    ex = shots_df.sample(n=min(k, len(shots_df)),
                         random_state=random.randint(0, 1e6))
    out = [
        "You are an assistant that decides which response style(s) "
        "the user needs. Possible styles: Information, Direct Guidance,"
        " Emotional Support.\n"
    ]
    for _, r in ex.iterrows():
        labels = ast.literal_eval(r["final_AS"]) if isinstance(r["final_AS"], str) else r["final_AS"]
        out.append(f"### Example\nUser: \"{r['question']}\"\n"
                   f"Needed Style(s): {', '.join(LABEL_DESC[l] for l in labels)}\n")
    out.append("### Task\n"
               f"User: \"{query}\"\n"
               "Needed Style(s):")
    return "\n".join(out)

def build_train_pair(row, shots_df):
    """Return prompt & target text (comma‑separated labels)."""
    labels = ast.literal_eval(row["final_AS"]) if isinstance(row["final_AS"], str) else row["final_AS"]
    tgt = ", ".join(LABEL_DESC[l] for l in labels)
    prmpt = build_prompt(row["question"], shots_df)
    return prmpt, tgt

# ------------------------------------------------------------------
# 3.  DATASET  →  input_ids / attention_mask / labels
# ------------------------------------------------------------------
def make_sft_dataset(df: pd.DataFrame, shots_df):
    records = []
    for _, row in df.iterrows():
        prompt, target = build_train_pair(row, shots_df)
        prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
        target_ids = tokenizer(" " + target + tokenizer.eos_token,
                               add_special_tokens=False).input_ids
        input_ids = prompt_ids + target_ids
        labels = [-100]*len(prompt_ids) + target_ids          # loss only on answer
        records.append({
            "input_ids": input_ids,
            "labels": labels,
            "attention_mask": [1]*len(input_ids)
        })
    return Dataset.from_list(records)

class PadCollator:
    """Pads input_ids, labels, attention_mask the same way."""
    def __init__(self, tok):
        self.tok = tok
    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            f["input_ids"]       += [self.tok.pad_token_id]*pad_len
            f["attention_mask"]  += [0]*pad_len
            f["labels"]          += [-100]*pad_len
        return {k: torch.tensor([f[k] for f in features]) for k in features[0]}

# ------------------------------------------------------------------
# 4.  METRIC  (generation + parsing)
# ------------------------------------------------------------------
def parse_styles(text: str):
    """Return set of label‑ids {'1','2'} found in generated text."""
    found = set()
    for lab, rgx in LABEL_REGEX.items():
        if rgx.search(text):
            found.add(lab)
    return found

def evaluate_generation(model, df_eval, shots_df):
    model.eval()
    preds_vec, gold_vec = [], []
    with torch.no_grad():
        for _, row in df_eval.iterrows():
            prompt = build_prompt(row["question"], shots_df)
            input_ids = tokenizer(prompt, return_tensors="pt").to(DEVICE)
            gen_ids = model.generate(
                **input_ids,
                max_new_tokens=15,
                do_sample=False
            )
            gen_text = tokenizer.decode(gen_ids[0][input_ids["input_ids"].shape[1]:],
                                        skip_special_tokens=True)
            pred = parse_styles(gen_text)
            gold = ast.literal_eval(row["final_AS"]) if isinstance(row["final_AS"], str) else row["final_AS"]
            preds_vec.append(multihot(pred))
            gold_vec.append(multihot(gold))

    preds_arr = np.array(preds_vec)
    gold_arr  = np.array(gold_vec)
    f1  = f1_score(gold_arr, preds_arr, average="macro", zero_division=0)
    jac = jaccard_score(gold_arr, preds_arr, average="samples", zero_division=0)
    return {"macro_f1": f1, "jaccard": jac}

# ------------------------------------------------------------------
# 5.  BUILD DATA  (assumes train_df, val_df, fewshot_sample exist)
# ------------------------------------------------------------------
train_dataset = make_sft_dataset(train_df, fewshot_sample)
val_dataset   = make_sft_dataset(val_df,   fewshot_sample)

collator = PadCollator(tokenizer)

# ------------------------------------------------------------------
# 6.  TRAIN
# ------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,          # effective batch ↑ without OOM
    learning_rate=2e-4,                     # LoRA often fine with 2e‑4
    lr_scheduler_type="cosine",
    eva_strategy="no",               # we’ll eval by generation later
    save_strategy="epoch",
    logging_steps=20,
    save_total_limit=2,
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator
)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# ------------------------------------------------------------
# TRAIN  ⟶  2 epochs  +  eval after each epoch
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,                 # ← two epochs
    evaluation_strategy="epoch",        # ← run eval at the end of every epoch
    save_strategy="epoch",              # (keeps best & checkpoints)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    seed=SEED,
    logging_steps=20,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,   # ← give it the validation split
    data_collator=collator
)



TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
trainer.train()



`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=10, training_loss=4.494994354248047, metrics={'train_runtime': 759.2369, 'train_samples_per_second': 0.722, 'train_steps_per_second': 0.013, 'total_flos': 1.299463230154752e+16, 'train_loss': 4.494994354248047, 'epoch': 2.0})

In [ ]:
metrics = evaluate_generation(model, val_df, fewshot_sample)
print(metrics)

In [ ]:
trainer.train()

# ------------------------------------------------------------------
# 7.  GENERATION‑TIME EVAL
# ------------------------------------------------------------------
metrics = evaluate_generation(model, val_df, fewshot_sample)
print(metrics)

In [ ]:
# ================================================================
# FEW-SHOT   MULTI-LABEL   STYLE-CLASSIFIER   (Qwen-3-8B)
# ================================================================
!pip install -q scikit-learn datasets

import os, ast, random, numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score, jaccard_score

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)

# ----------------------------------------------------------
# 0. ENV & CONSTANTS
# ----------------------------------------------------------
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

SEED            = 42
FEWSHOT_K       = 3                      # ↔ how many examples to prepend
BASE_MODEL      = "Qwen/Qwen3-8B"        # chat version works fine
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LABEL_ID   = {"1": 0, "2": 1, "3": 2}
LABEL_DESC = {"1": "Information",
              "2": "Direct Guidance",
              "3": "Emotional Support"}

# tokenizer = AutoTokenizer.from_pretrained(
#     BASE_MODEL,
#     trust_remote_code=True
# )

# ----------------------------------------------------------
# 1.  UTILS
# ----------------------------------------------------------
def multihot(label_list):
    """Convert list of str → multi-hot vec."""
    y = [0]*len(LABEL_ID)
    for lab in label_list:
        if lab in LABEL_ID: y[LABEL_ID[lab]] = 1
    return y

def build_prompt(query, shots_df, k=FEWSHOT_K):
    """Few-shot prompt with *k* exemplars."""
    exemplars = shots_df.sample(
        n=min(k, len(shots_df)),
        random_state=random.randint(0, 10**6)
    )

    parts = [
      "You are an assistant that decides which response style(s)"
      " the user needs. Possible styles are: "
      "Information, Direct Guidance, Emotional Support.\n"
    ]

    # ----- exemplars -----
    for _, row in exemplars.iterrows():
        lbls = ast.literal_eval(row["final_AS"]) if isinstance(row["final_AS"], str) else row["final_AS"]
        lbl_names = ", ".join(LABEL_DESC[l] for l in lbls)
        parts.append(
            f"### Example\nUser: \"{row['question']}\"\n"
            f"Needed Style(s): {lbl_names}\n"
        )

    # ----- target -----
    parts.append(
        "### Task\n"
        f"User: \"{query}\"\n"
        "Needed Style(s):"
    )
    return "\n".join(parts)

def make_hf_dataset(df, shot_df):
    """HF Dataset → {text, labels}"""
    texts, ys = [], []
    for _, row in df.iterrows():
        label_raw = ast.literal_eval(row["final_AS"]) if isinstance(row["final_AS"], str) else row["final_AS"]
        texts.append(build_prompt(row["question"], shot_df))
        ys.append(multihot(label_raw))
    return Dataset.from_dict({"text": texts, "labels": ys})

# ----------------------------------------------------------
# 2.  DATA
# ----------------------------------------------------------
# ▸ ASSUMPTION: train_df, val_df, fewshot_sample already exist
train_dataset = make_hf_dataset(train_df, fewshot_sample)
val_dataset   = make_hf_dataset(val_df,   fewshot_sample)

def tok_fn(batch):
    return tokenizer(batch["text"],
                     truncation=True,
                     padding="max_length",
                     max_length=768)

train_dataset = train_dataset.map(tok_fn, batched=True, remove_columns=["text"])
val_dataset   = val_dataset.map(tok_fn,   batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")

# ----------------------------------------------------------
# 3.  MODEL
# ----------------------------------------------------------
# model = AutoModelForSequenceClassification.from_pretrained(
#             BASE_MODEL,
#             num_labels=len(LABEL_ID),
#             problem_type="multi_label_classification",
#             trust_remote_code=True
#         ).to(DEVICE)


In [ ]:

# ----------------------------------------------------------
# 4.  METRICS
# ----------------------------------------------------------
def compute_metrics(eval_pred):
    logits, gold = eval_pred
    probs = 1 / (1 + np.exp(-logits))      # sigmoid
    preds = (probs >= 0.5).astype(int)

    f1  = f1_score(gold, preds, average="macro", zero_division=0)
    jac = jaccard_score(gold, preds, average="samples", zero_division=0)
    return {"macro_f1": f1, "jaccard": jac}

# ----------------------------------------------------------
# 5.  TRAIN ARGS
# ----------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=1,     # Qwen-8B is heavy → keep small
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=20,
    seed=SEED,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

# ----------------------------------------------------------
# 6.  TRAINER
# ----------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# ----------------------------------------------------------
# 7.  LAUNCH TRAINING
# ----------------------------------------------------------
trainer.train()


In [ ]:
def build_prompt(question: str) -> str:
    """
    Build a classification prompt asking the model to choose one response style
    from: Information, Direct Guidance, Emotional Support.
    """
    return (
        "You are an assistant that classifies user inputs by the style of response they require. "
        "Choose exactly one of: Information, Direct Guidance, Emotional Support.\n"
        f"User Input: \"{question}\"\n"
        "Response Style:"
    )


In [ ]:
train_dataset['text'][0]

'You are an assistant that classifies user inputs by the style of response they require. Choose exactly one of: Information, Direct Guidance, Emotional Support.\nUser Input: "افكر كثير لدرجه تنسيني مافعلت او ماذا سوف افعل كثرة القلق والارق المزمن والشعور بالسلبيه والخوف قد يصل الم جسدي"\nResponse Style: Information, Direct Guidance'

In [ ]:
val_dataset['text'][0]

'You are an assistant that classifies user inputs by the style of response they require. Choose exactly one of: Information, Direct Guidance, Emotional Support.\nUser Input: "اصبت باكتئاب ولادة منذ ٣سنوات و تعالجت و في حملي الثاني عاد الاكتئاب و الخوف و تعالجت ايضا مع العلم ان الطبيب لم يوقف العلاج من الاكتئاب الماضي هل سأحتاج طول عمري علاج للاكتئا"\nResponse Style: Information'

In [ ]:
import os

# Disable wandb BEFORE importing Trainer
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"  # extra safety

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
import torch
import ast

QT_LABELS = {
    "1": "Information",
    "2": "Direct Guidance",
    "3": "Emotional Support"
}

# ====================
# 1. Prepare data
# ====================

def format_for_lm(df):
    texts = []
    for _, row in df.iterrows():
        labels = row["final_AS"]
        if isinstance(labels, str):
            labels = ast.literal_eval(labels)

        label_desc = ', '.join([QT_LABELS.get(l, l) for l in labels])
        prompt = build_prompt(row["question"]) + " " + label_desc
        texts.append(prompt)
    return Dataset.from_dict({"text": texts})

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

# Convert pandas → HF Dataset
train_dataset = format_for_lm(train_df)
val_dataset = format_for_lm(val_df)

# Tokenize using HF Dataset
train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset = val_dataset.map(tokenize_fn, batched=True)

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ====================
# 2. Training Arguments
# ====================
# training_args = TrainingArguments(
#     output_dir="./results",
#     eval_strategy="epoch",       # <-- fixed key
#     save_strategy="epoch",
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     num_train_epochs=1,
#     logging_dir="./logs",
#     logging_steps=1,
#     save_total_limit=1,
#     load_best_model_at_end=True,
#     metric_for_best_model="loss",
#     greater_is_better=False,
#     report_to="none"  # <-- disables W&B and other loggers
# )

training_args = TrainingArguments(
    output_dir="./results",              # Where to save checkpoints
    eval_strategy="epoch",         # Evaluate every epoch
    save_strategy="epoch",               # Save every epoch
    per_device_train_batch_size=8,       # Increase for real training
    per_device_eval_batch_size=8,
    num_train_epochs=2,                  # Typical starting point
    learning_rate=2e-5,                  # Good default for LoRA fine-tuning
    weight_decay=0.01,                   # Regularization
    logging_dir="./logs",                # TensorBoard/W&B logs
    logging_steps=50,                    # Log every 50 steps
    save_total_limit=2,                  # Keep last 2 checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="loss",        # Could change to F1 later
    greater_is_better=False,
    report_to="none"
)

# ====================
# 3. Metrics
# ====================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    mask = labels != -100
    labels = labels[mask]
    preds = preds[mask]

    f1 = f1_score(labels, preds, average="macro")
    jac = jaccard_score(labels, preds, average="macro")

    return {"f1": f1, "jaccard": jac}

# ====================
# 4. Trainer
# ====================
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# ====================
# 5. Train (tiny run)
# ====================

Map:   0%|          | 0/272 [00:00<?, ? examples/s]

Map:   0%|          | 0/68 [00:00<?, ? examples/s]

/tmp/ipython-input-59-914821435.py:107: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,F1,Jaccard
1,No log,3.313740,0.000566,0.000324
2,3.451800,3.174215,0.000545,0.000313


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=68, training_loss=3.388732349171358, metrics={'train_runtime': 425.8355, 'train_samples_per_second': 1.277, 'train_steps_per_second': 0.16, 'total_flos': 6330445562118144.0, 'train_loss': 3.388732349171358, 'epoch': 2.0})

In [ ]:
# ================================================================
# FEW-SHOT   MULTI-LABEL   STYLE-CLASSIFIER   (Qwen-3-8B)
# ================================================================
# !pip install -q scikit-learn datasets

import os, ast, random, numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import f1_score, jaccard_score

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)

# ----------------------------------------------------------
# 0. ENV & CONSTANTS
# ----------------------------------------------------------
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

SEED            = 42
FEWSHOT_K       = 3                      # ↔ how many examples to prepend
BASE_MODEL      = "Qwen/Qwen3-8B"        # chat version works fine
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LABEL_ID   = {"1": 0, "2": 1, "3": 2}
LABEL_DESC = {"1": "Information",
              "2": "Direct Guidance",
              "3": "Emotional Support"}

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

# ----------------------------------------------------------
# 1.  UTILS
# ----------------------------------------------------------
def multihot(label_list):
    """Convert list of str → multi-hot vec."""
    y = [0]*len(LABEL_ID)
    for lab in label_list:
        if lab in LABEL_ID: y[LABEL_ID[lab]] = 1
    return y

def build_prompt(query, shots_df, k=FEWSHOT_K):
    """Few-shot prompt with *k* exemplars."""
    exemplars = shots_df.sample(
        n=min(k, len(shots_df)),
        random_state=random.randint(0, 10**6)
    )

    parts = [
      "You are an assistant that decides which response style(s)"
      " the user needs. Possible styles are: "
      "Information, Direct Guidance, Emotional Support.\n"
    ]

    # ----- exemplars -----
    for _, row in exemplars.iterrows():
        lbls = ast.literal_eval(row["final_AS"]) if isinstance(row["final_AS"], str) else row["final_AS"]
        lbl_names = ", ".join(LABEL_DESC[l] for l in lbls)
        parts.append(
            f"### Example\nUser: \"{row['question']}\"\n"
            f"Needed Style(s): {lbl_names}\n"
        )

    # ----- target -----
    parts.append(
        "### Task\n"
        f"User: \"{query}\"\n"
        "Needed Style(s):"
    )
    return "\n".join(parts)

def make_hf_dataset(df, shot_df):
    """HF Dataset → {text, labels}"""
    texts, ys = [], []
    for _, row in df.iterrows():
        label_raw = ast.literal_eval(row["final_AS"]) if isinstance(row["final_AS"], str) else row["final_AS"]
        texts.append(build_prompt(row["question"], shot_df))
        ys.append(multihot(label_raw))
    return Dataset.from_dict({"text": texts, "labels": ys})

# ----------------------------------------------------------
# 2.  DATA
# ----------------------------------------------------------
# ▸ ASSUMPTION: train_df, val_df, fewshot_sample already exist
train_dataset = make_hf_dataset(train_df, fewshot_sample)
val_dataset   = make_hf_dataset(val_df,   fewshot_sample)


In [ ]:

def tok_fn(batch):
    return tokenizer(batch["text"],
                     truncation=True,
                     padding="max_length",
                     max_length=768)

train_dataset = train_dataset.map(tok_fn, batched=True, remove_columns=["text"])
val_dataset   = val_dataset.map(tok_fn,   batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")

In [ ]:
print(train_dataset['text'][0])

You are an assistant that decides which response style(s) the user needs. Possible styles are: Information, Direct Guidance, Emotional Support.

### Example
User: "ضيقة. قلة تركيز. صداع مستمر. توتر. خوف. كثرة التفكير. الاغتراب عن النفس. أحيانا آلام في الصدر. كره التجمعات. الانفعال. الغضب. الإرهاق والتعب. ضعف في ثقة بالنفس و تقدير الذات"
Needed Style(s): Information

### Example
User: "قلق دائم و اثناء القلق تسدد في الشهية و خفقان في القلب و ونوبات هلع و الشعور بالغرابة وسواس ٢٤ ساعة و سواس اثناء التكلم ذا كله بسبب بحثي عنه الله لا يبلاني فيه يارب ساعدوني"
Needed Style(s): Information

### Example
User: "بقالي فترة طويلة دايما مضايق و حاسس بخنقةو قلق و عدم قدرة علي التركيز و يغلب عليهم شغور بالخمول و فقد الشغف حتي في الشغل حالتي بتسوق اكتر كل فترة و مش عارف احس بأي حاجة حواليا"
Needed Style(s): Information

### Task
User: "افكر كثير لدرجه تنسيني مافعلت او ماذا سوف افعل كثرة القلق والارق المزمن والشعور بالسلبيه والخوف قد يصل الم جسدي"
Needed Style(s):


In [ ]:
train_dataset['labels']

Column([[1, 1, 0], [1, 1, 1], [1, 1, 0], [1, 1, 0], [1, 0, 0]])

In [ ]:


# ----------------------------------------------------------
# 3.  MODEL
# ----------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL,
            num_labels=len(LABEL_ID),
            problem_type="multi_label_classification",
            trust_remote_code=True
        ).to(DEVICE)

# ----------------------------------------------------------
# 4.  METRICS
# ----------------------------------------------------------
def compute_metrics(eval_pred):
    logits, gold = eval_pred
    probs = 1 / (1 + np.exp(-logits))      # sigmoid
    preds = (probs >= 0.5).astype(int)

    f1  = f1_score(gold, preds, average="macro", zero_division=0)
    jac = jaccard_score(gold, preds, average="samples", zero_division=0)
    return {"macro_f1": f1, "jaccard": jac}

# ----------------------------------------------------------
# 5.  TRAIN ARGS
# ----------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=1,     # Qwen-8B is heavy → keep small
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=20,
    seed=SEED,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

# ----------------------------------------------------------
# 6.  TRAINER
# ----------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# ----------------------------------------------------------
# 7.  LAUNCH TRAINING
# ----------------------------------------------------------
trainer.train()


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/272 [00:00<?, ? examples/s]

Map:   0%|          | 0/68 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# Ensure QT_LABELS and QT_DESCRIPTIONS are consistent
QT_LABELS = {
    "A": "Diagnosis",
    "B": "Treatment",
    "C": "Anatomy and physiology",
    "D": "Epidemiology",
    "E": "Healthy lifestyle",
    "F": "Provider choices",
    "Z": "Other"
}

QT_DESCRIPTIONS = QT_LABELS  # Use same descriptions for inference

# Load test file
test_path = "/content/drive/MyDrive/MentalQA-/subtask1_input_test.tsv"
test_df = pd.read_csv(test_path, sep="\t", header=None, names=["question"])

# Predict
predictions = []
for q in test_df["question"]:
    prompt = build_fewshot_prompt(q, QT_DESCRIPTIONS)
    output = pipe(prompt, max_new_tokens=50, do_sample=False)[0]['generated_text']
    predicted_label = output.split("التصنيف:")[-1].strip()
    predictions.append(predicted_label)

# Save predictions
test_df["predictions"] = predictions
output_path = "/content/drive/MyDrive/MentalQA-/subtask1_predictions.tsv"
test_df.to_csv(output_path, sep="\t", index=False)

print(f"Predictions saved to: {output_path}")


In [ ]:
# # 🧪 Test on first training question
# sample_question = train_df.iloc[0]["question"]
# prompt = build_prompt(sample_question)

# # 🔍 Generate prediction
# output = pipe(prompt)[0]['generated_text']
# print("🔹 Prompt:\n", prompt)
# print("\n🔸 Model Output:\n", output)


🔹 Prompt:
 Q: عزباء ٢٢ مشكلتي يجيني شعور غريب بان أوذي نفسي مثلا بالسيارة وهي سريعة يجيني شعور أفتح الباب وأرمي نفسي أو أشق يدي بالسكين اذا شفت مرض اخاف منه بس بعقلي يقول يارب يجيني بعدين استغفر مرات يجيني شعور أأذي اللي أحبهم مرات لمن أكون بالمول بالدور الثاني أخاف من المرتفع بس يجيني شعور أرمي نفسي بعد تعبت
A: Diagnosis

Q: قلق دائم و اثناء القلق تسدد في الشهية و خفقان في القلب و ونوبات هلع و الشعور بالغرابة وسواس ٢٤ ساعة و سواس اثناء التكلم ذا كله بسبب بحثي عنه الله لا يبلاني فيه يارب ساعدوني
A: Treatment

Q: انا فتاة اشعر انني مقبلة في كثير من الاحيان علي الانتحار، انا الان اعيش وحدي في بلد غريب اشعر بأن كل من حولي يريدون ايذائي، اشعر بأن عائلتي لا تحبني بل يتحدثون معي للشفقة، اعاني من الكثير لا يمكنني وصفه هنا
A: Treatment

Q: اعاني من عدم التحكم بغضبي وعند ضرب شخص اضرب وانا لستو بوعييي لدرجة انني لااشعر بالذي افعله لكن اشعر وكانني اتلذذ بالضرب بعد الضرب استوعب ماذا فعلت وابكي واضرب نفسي واشعر بكتمه
A: Diagnosis, Healthy lifestyle

Q: بدأت مشكلتي قبل 5 سنوات عندما اكتشفت انه عندي 